# m05 · Dòng chảy nén được: các phép tính kiểm chứng

Sổ tay này tính lại bằng Python các ví dụ chính của module m05 (Slide Ch.5; Cengel Ch.17 §17-1…17-4). Mỗi mục có ba phần: câu hỏi mà phương pháp trả lời → mã → diễn giải kết quả THẬT. Các bảng cần dùng (A-32, A-17, A-20) nhúng trong ô mã đầu tiên. Đường A là công thức đóng của slide; đường B là cách số độc lập (cực đại số của ρV, ODE Hugoniot, đạo hàm số của P(ρ), nội suy bảng). Slide Ch.5 dừng ở vòi hội tụ: sóng xung kích, Fanno, Rayleigh CHƯA LÀM.

## 1. Tính chất hãm: slide tr.10–13 (Cengel Ví dụ 17–1)

**Câu hỏi:** không khí 250 m/s ở 5000 m (54,05 kPa; 255,7 K) bị hãm đẳng entropi trong diffuser rồi nén với tỉ số hãm 8: T₀₁, P₀₁, T₀₂ và công nén là bao nhiêu? Công thức đóng so với tích phân số dT = −V dV/c_p và w = ∫RT d ln P.

In [1]:
import math, numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq, minimize_scalar
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A32 = np.array([[0.1, 0.1094, 5.8218, 0.993, 0.995, 0.998], [0.2, 0.2182, 2.9635, 0.9725, 0.9803, 0.9921], [0.3, 0.3257, 2.0351, 0.9395, 0.9564, 0.9823], [0.4, 0.4313, 1.5901, 0.8956, 0.9243, 0.969], [0.5, 0.5345, 1.3398, 0.843, 0.8852, 0.9524], [0.6, 0.6348, 1.1882, 0.784, 0.8405, 0.9328], [0.7, 0.7318, 1.0944, 0.7209, 0.7916, 0.9107], [0.8, 0.8251, 1.0382, 0.656, 0.74, 0.8865], [0.9, 0.9146, 1.0089, 0.5913, 0.687, 0.8606], [1.0, 1.0, 1.0, 0.5283, 0.6339, 0.8333], [1.2, 1.1583, 1.0304, 0.4124, 0.5311, 0.7764], [1.4, 1.2999, 1.1149, 0.3142, 0.4374, 0.7184], [1.6, 1.4254, 1.2502, 0.2353, 0.3557, 0.6614], [1.8, 1.536, 1.439, 0.174, 0.2868, 0.6068], [2.0, 1.633, 1.6875, 0.1278, 0.23, 0.5556], [2.2, 1.7179, 2.005, 0.0935, 0.1841, 0.5081], [2.4, 1.7922, 2.4031, 0.0684, 0.1472, 0.4647], [2.6, 1.8571, 2.896, 0.0501, 0.1179, 0.4252], [2.8, 1.914, 3.5001, 0.0368, 0.0946, 0.3894], [3.0, 1.964, 4.2346, 0.0272, 0.076, 0.3571], [5.0, 2.2361, 25.0, 0.0019, 0.0113, 0.1667]])     # A-32 (k = 1.4) rows: Ma, Ma*, A/A*, P/P0, rho/rho0, T/T0
A17 = np.array([[250.0, 250.05, 0.7329], [260.0, 260.09, 0.8405], [270.0, 270.11, 0.959], [280.0, 280.13, 1.0889], [285.0, 285.14, 1.1584], [290.0, 290.16, 1.2311], [295.0, 295.17, 1.3068], [298.0, 298.18, 1.3543], [300.0, 300.19, 1.386], [305.0, 305.22, 1.4686], [310.0, 310.24, 1.5546], [315.0, 315.27, 1.6442], [320.0, 320.29, 1.7375], [325.0, 325.31, 1.8345], [330.0, 330.34, 1.9352], [340.0, 340.42, 2.149], [350.0, 350.49, 2.379], [360.0, 360.58, 2.626], [370.0, 370.67, 2.892], [380.0, 380.77, 3.176], [390.0, 390.88, 3.481], [400.0, 400.98, 3.806], [410.0, 411.12, 4.153], [420.0, 421.26, 4.522], [430.0, 431.43, 4.915], [440.0, 441.61, 5.332], [450.0, 451.8, 5.775], [460.0, 462.02, 6.245], [470.0, 472.24, 6.742], [480.0, 482.49, 7.268], [490.0, 492.74, 7.824], [500.0, 503.02, 8.411], [510.0, 513.32, 9.031], [520.0, 523.63, 9.684], [530.0, 533.98, 10.37], [540.0, 544.35, 11.1], [550.0, 555.74, 11.86], [560.0, 565.17, 12.66], [570.0, 575.59, 13.5], [580.0, 586.04, 14.38], [590.0, 596.52, 15.31], [600.0, 607.02, 16.28], [610.0, 617.53, 17.3], [620.0, 628.07, 18.36], [630.0, 638.63, 19.84], [640.0, 649.22, 20.64], [650.0, 659.84, 21.86], [660.0, 670.47, 23.13], [670.0, 681.14, 24.46], [680.0, 691.82, 25.85], [690.0, 702.52, 27.29], [700.0, 713.27, 28.8], [710.0, 724.04, 30.38], [720.0, 734.82, 32.02], [730.0, 745.62, 33.72], [740.0, 756.44, 35.5], [750.0, 767.29, 37.35], [760.0, 778.18, 39.27], [780.0, 800.03, 43.35], [800.0, 821.95, 47.75]])     # A-17 air: T [K], h [kJ/kg], Pr
A20 = np.array([[300.0, 9431.0], [310.0, 9807.0], [320.0, 10186.0], [330.0, 10570.0], [340.0, 10959.0], [350.0, 11351.0], [360.0, 11748.0], [370.0, 12148.0], [380.0, 12552.0], [390.0, 12960.0], [400.0, 13372.0], [410.0, 13787.0], [420.0, 14206.0], [430.0, 14628.0], [440.0, 15054.0], [450.0, 15483.0], [460.0, 15916.0], [470.0, 16351.0], [480.0, 16791.0], [490.0, 17232.0], [500.0, 17678.0], [510.0, 18126.0], [520.0, 18576.0], [530.0, 19029.0], [540.0, 19485.0], [550.0, 19945.0], [560.0, 20407.0], [570.0, 20870.0], [580.0, 21337.0], [590.0, 21807.0], [600.0, 22280.0], [610.0, 22754.0], [620.0, 23231.0], [630.0, 23709.0], [640.0, 24190.0], [650.0, 24674.0], [660.0, 25160.0], [670.0, 25648.0], [680.0, 26138.0], [690.0, 26631.0], [700.0, 27125.0], [710.0, 27622.0], [720.0, 28121.0], [730.0, 28622.0], [740.0, 29124.0], [750.0, 29629.0], [760.0, 30135.0], [770.0, 30644.0], [780.0, 31154.0], [790.0, 31665.0], [800.0, 32179.0], [810.0, 32694.0], [820.0, 33212.0], [830.0, 33730.0], [840.0, 34251.0], [850.0, 34773.0], [860.0, 35296.0], [870.0, 35821.0], [880.0, 36347.0], [890.0, 36876.0], [900.0, 37405.0], [910.0, 37935.0], [920.0, 38467.0], [930.0, 39000.0], [940.0, 39535.0], [950.0, 40070.0], [960.0, 40607.0], [970.0, 41145.0], [980.0, 41685.0], [990.0, 42226.0], [1000.0, 42769.0], [1020.0, 43859.0], [1040.0, 44953.0], [1060.0, 46051.0], [1080.0, 47153.0], [1100.0, 48258.0], [1120.0, 49369.0], [1140.0, 50484.0], [1160.0, 51602.0], [1180.0, 52724.0], [1200.0, 53848.0], [1220.0, 54977.0], [1240.0, 56108.0], [1260.0, 57244.0], [1280.0, 58381.0], [1300.0, 59522.0], [1320.0, 60666.0], [1340.0, 61813.0], [1360.0, 62963.0], [1380.0, 64116.0], [1400.0, 65271.0]])     # A-20 CO2: T [K], h [kJ/kmol]
# ---- route A: closed forms of the slide equations (SI: P kPa, R kJ/(kg K), V m/s, A m2, mdot kg/s)
def cp_of(k, R): return k * R / (k - 1.0)
def T0_of(T, V, cp): return T + V * V / (2000.0 * cp)                         # (1)/(7): V^2/2 in J/kg -> kJ/kg
def P0_of(P, T, T0, k): return P * (T0 / T) ** (k / (k - 1.0))               # (2)
def csound(k, R, T): return math.sqrt(k * R * 1000.0 * T)
def TT0(Ma, k): return 1.0 / (1.0 + (k - 1.0) / 2.0 * Ma * Ma)               # (9)
def PP0(Ma, k): return TT0(Ma, k) ** (k / (k - 1.0))                         # (10)
def AAs(Ma, k): return (1.0 / Ma) * ((2.0 / (k + 1.0)) * (1.0 + (k - 1.0) / 2.0 * Ma * Ma)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (17)
def crit(k): return dict(TT=2.0 / (k + 1.0), PP=(2.0 / (k + 1.0)) ** (k / (k - 1.0)))                                      # (12),(13)
def Ma_of_PP(pr, k): return math.sqrt(2.0 / (k - 1.0) * (pr ** (-(k - 1.0) / k) - 1.0))
def mdot_choked(A, P0, T0, k, R): return A * (P0 * 1000.0) * math.sqrt(k / (R * 1000.0 * T0)) * (2.0 / (k + 1.0)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (16) with P0 in Pa, R in J/(kg K)
# ---- route B: numerical descriptions that do not use (9)-(17)
def flux_max(k):
    """x* = T/T0 maximising the mass flux rho V ~ x^(1/(k-1)) sqrt(1-x) along the isentrope (energy equation V = sqrt(2 cp (T0 - T)))."""
    r = minimize_scalar(lambda x: -(x ** (1.0 / (k - 1.0)) * math.sqrt(1.0 - x)), bounds=(1e-6, 1 - 1e-9), method='bounded', options=dict(xatol=1e-13)); return r.x, -r.fun
def isentrope_T(T, P, Pnew, k):
    s = solve_ivp(lambda x, y: [(k - 1.0) / k], [0.0, math.log(Pnew / P)], [math.log(T)], rtol=1e-11, atol=1e-13); return math.exp(s.y[0, -1])
def c_num(T, k, R):
    """speed of sound as (dP/d rho)_s by central difference on the isentrope P = C rho^k (not sqrt(kRT))."""
    P0_ = 1.0e5; rho = P0_ / (R * 1000.0 * T); C = P0_ / rho ** k; h = 1e-5 * rho; return math.sqrt((C * (rho + h) ** k - C * (rho - h) ** k) / (2 * h))
def x_of_Ma(Ma, k, R=0.287):
    cp = cp_of(k, R); return brentq(lambda T: Ma * c_num(T, k, R) - math.sqrt(max(2000.0 * cp * (1000.0 - T), 0.0)), 1.0, 1000.0 - 1e-9, xtol=1e-12) / 1000.0
T, P, V, cp, k, R = 255.7, 54.05, 250.0, 1.005, 1.4, 0.287
T01 = T0_of(T, V, cp); P01 = P0_of(P, T, T01, k); T02 = T01 * 8 ** ((k - 1) / k); w = cp * (T02 - T01)
print(f"route A: T01 = {T01:.2f} K; P01 = {P01:.2f} kPa; T02 = {T02:.1f} K; w = {w:.1f} kJ/kg   (slide: 286.8; 80.77; 519.5; 233.9)")
s = solve_ivp(lambda v, y: [-v / (1000.0 * cp)], [V, 0.0], [T], rtol=1e-11, atol=1e-11); T01b = s.y[0, -1]
P01b = P * math.exp(quad(lambda t: (k / (k - 1)) / t, T, T01b)[0])                       # ds = 0: d ln P = k/(k-1) d ln T
wb = abs(quad(lambda x: R * isentrope_T(T01b, 1.0, math.exp(x), k), 0.0, math.log(8.0))[0])
print(f"route B: T01 = {T01b:.2f} K; P01 = {P01b:.2f} kPa; w = {wb:.1f} kJ/kg  (cp = kR/(k-1) = {cp_of(k, R):.4f} instead of 1.005: {100 * (wb / w - 1):+.2f} %)")
print("ram rise:", f"dT = {T01 - T:.1f} K, dP = {P01 - P:.2f} kPa")
for Vp in (1, 10, 100, 1000): print(f"probe in air at 300 K, V = {Vp:4d} m/s reads {T0_of(300.0, Vp, cp):.2f} K")

route A: T01 = 286.79 K; P01 = 80.77 kPa; T02 = 519.5 K; w = 233.9 kJ/kg   (slide: 286.8; 80.77; 519.5; 233.9)
route B: T01 = 286.79 K; P01 = 80.77 kPa; w = 233.8 kJ/kg  (cp = kR/(k-1) = 1.0045 instead of 1.005: -0.05 %)
ram rise: dT = 31.1 K, dP = 26.72 kPa
probe in air at 300 K, V =    1 m/s reads 300.00 K
probe in air at 300 K, V =   10 m/s reads 300.05 K
probe in air at 300 K, V =  100 m/s reads 304.98 K
probe in air at 300 K, V = 1000 m/s reads 797.51 K


**Đọc kết quả:** T₀₁ = 286,79 K, P₀₁ = 80,77 kPa, T₀₂ = 519,5 K, công 233,9 kJ/kg khớp slide (286,8; 80,77; 519,5; 233,9). Đường B (tích phân số) cho cùng T₀₁, P₀₁; công chênh -0,05 % vì kR/(k−1) = 1,0045 chứ không phải 1,005. Hãm từ 250 m/s làm tăng 31,1 K và 26,72 kPa. Đầu dò ở 100 m/s chỉ 304,98 K, ở 1000 m/s chỉ 797,51 K (slide tr.14 không in đáp án; c_p không đổi là xấp xỉ thô ở 1000 m/s).

## 2. Công máy nén đẳng entropi: slide tr.15 (Cengel 17–8, đáp án in 5,27 kW) và A-17

**Câu hỏi:** không khí vào máy nén ở T₀₁ = 27 °C, tỉ số áp suất hãm 9, 0,02 kg/s: công suất là bao nhiêu và c_p biến thiên (bảng A-17, P_r) thay đổi nó bao nhiêu? Đây là bài có đáp án in dùng để thử phương pháp trước khi áp dụng cho bài không đáp án (tr.16).

In [2]:
import math, numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq, minimize_scalar
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A32 = np.array([[0.1, 0.1094, 5.8218, 0.993, 0.995, 0.998], [0.2, 0.2182, 2.9635, 0.9725, 0.9803, 0.9921], [0.3, 0.3257, 2.0351, 0.9395, 0.9564, 0.9823], [0.4, 0.4313, 1.5901, 0.8956, 0.9243, 0.969], [0.5, 0.5345, 1.3398, 0.843, 0.8852, 0.9524], [0.6, 0.6348, 1.1882, 0.784, 0.8405, 0.9328], [0.7, 0.7318, 1.0944, 0.7209, 0.7916, 0.9107], [0.8, 0.8251, 1.0382, 0.656, 0.74, 0.8865], [0.9, 0.9146, 1.0089, 0.5913, 0.687, 0.8606], [1.0, 1.0, 1.0, 0.5283, 0.6339, 0.8333], [1.2, 1.1583, 1.0304, 0.4124, 0.5311, 0.7764], [1.4, 1.2999, 1.1149, 0.3142, 0.4374, 0.7184], [1.6, 1.4254, 1.2502, 0.2353, 0.3557, 0.6614], [1.8, 1.536, 1.439, 0.174, 0.2868, 0.6068], [2.0, 1.633, 1.6875, 0.1278, 0.23, 0.5556], [2.2, 1.7179, 2.005, 0.0935, 0.1841, 0.5081], [2.4, 1.7922, 2.4031, 0.0684, 0.1472, 0.4647], [2.6, 1.8571, 2.896, 0.0501, 0.1179, 0.4252], [2.8, 1.914, 3.5001, 0.0368, 0.0946, 0.3894], [3.0, 1.964, 4.2346, 0.0272, 0.076, 0.3571], [5.0, 2.2361, 25.0, 0.0019, 0.0113, 0.1667]])     # A-32 (k = 1.4) rows: Ma, Ma*, A/A*, P/P0, rho/rho0, T/T0
A17 = np.array([[250.0, 250.05, 0.7329], [260.0, 260.09, 0.8405], [270.0, 270.11, 0.959], [280.0, 280.13, 1.0889], [285.0, 285.14, 1.1584], [290.0, 290.16, 1.2311], [295.0, 295.17, 1.3068], [298.0, 298.18, 1.3543], [300.0, 300.19, 1.386], [305.0, 305.22, 1.4686], [310.0, 310.24, 1.5546], [315.0, 315.27, 1.6442], [320.0, 320.29, 1.7375], [325.0, 325.31, 1.8345], [330.0, 330.34, 1.9352], [340.0, 340.42, 2.149], [350.0, 350.49, 2.379], [360.0, 360.58, 2.626], [370.0, 370.67, 2.892], [380.0, 380.77, 3.176], [390.0, 390.88, 3.481], [400.0, 400.98, 3.806], [410.0, 411.12, 4.153], [420.0, 421.26, 4.522], [430.0, 431.43, 4.915], [440.0, 441.61, 5.332], [450.0, 451.8, 5.775], [460.0, 462.02, 6.245], [470.0, 472.24, 6.742], [480.0, 482.49, 7.268], [490.0, 492.74, 7.824], [500.0, 503.02, 8.411], [510.0, 513.32, 9.031], [520.0, 523.63, 9.684], [530.0, 533.98, 10.37], [540.0, 544.35, 11.1], [550.0, 555.74, 11.86], [560.0, 565.17, 12.66], [570.0, 575.59, 13.5], [580.0, 586.04, 14.38], [590.0, 596.52, 15.31], [600.0, 607.02, 16.28], [610.0, 617.53, 17.3], [620.0, 628.07, 18.36], [630.0, 638.63, 19.84], [640.0, 649.22, 20.64], [650.0, 659.84, 21.86], [660.0, 670.47, 23.13], [670.0, 681.14, 24.46], [680.0, 691.82, 25.85], [690.0, 702.52, 27.29], [700.0, 713.27, 28.8], [710.0, 724.04, 30.38], [720.0, 734.82, 32.02], [730.0, 745.62, 33.72], [740.0, 756.44, 35.5], [750.0, 767.29, 37.35], [760.0, 778.18, 39.27], [780.0, 800.03, 43.35], [800.0, 821.95, 47.75]])     # A-17 air: T [K], h [kJ/kg], Pr
A20 = np.array([[300.0, 9431.0], [310.0, 9807.0], [320.0, 10186.0], [330.0, 10570.0], [340.0, 10959.0], [350.0, 11351.0], [360.0, 11748.0], [370.0, 12148.0], [380.0, 12552.0], [390.0, 12960.0], [400.0, 13372.0], [410.0, 13787.0], [420.0, 14206.0], [430.0, 14628.0], [440.0, 15054.0], [450.0, 15483.0], [460.0, 15916.0], [470.0, 16351.0], [480.0, 16791.0], [490.0, 17232.0], [500.0, 17678.0], [510.0, 18126.0], [520.0, 18576.0], [530.0, 19029.0], [540.0, 19485.0], [550.0, 19945.0], [560.0, 20407.0], [570.0, 20870.0], [580.0, 21337.0], [590.0, 21807.0], [600.0, 22280.0], [610.0, 22754.0], [620.0, 23231.0], [630.0, 23709.0], [640.0, 24190.0], [650.0, 24674.0], [660.0, 25160.0], [670.0, 25648.0], [680.0, 26138.0], [690.0, 26631.0], [700.0, 27125.0], [710.0, 27622.0], [720.0, 28121.0], [730.0, 28622.0], [740.0, 29124.0], [750.0, 29629.0], [760.0, 30135.0], [770.0, 30644.0], [780.0, 31154.0], [790.0, 31665.0], [800.0, 32179.0], [810.0, 32694.0], [820.0, 33212.0], [830.0, 33730.0], [840.0, 34251.0], [850.0, 34773.0], [860.0, 35296.0], [870.0, 35821.0], [880.0, 36347.0], [890.0, 36876.0], [900.0, 37405.0], [910.0, 37935.0], [920.0, 38467.0], [930.0, 39000.0], [940.0, 39535.0], [950.0, 40070.0], [960.0, 40607.0], [970.0, 41145.0], [980.0, 41685.0], [990.0, 42226.0], [1000.0, 42769.0], [1020.0, 43859.0], [1040.0, 44953.0], [1060.0, 46051.0], [1080.0, 47153.0], [1100.0, 48258.0], [1120.0, 49369.0], [1140.0, 50484.0], [1160.0, 51602.0], [1180.0, 52724.0], [1200.0, 53848.0], [1220.0, 54977.0], [1240.0, 56108.0], [1260.0, 57244.0], [1280.0, 58381.0], [1300.0, 59522.0], [1320.0, 60666.0], [1340.0, 61813.0], [1360.0, 62963.0], [1380.0, 64116.0], [1400.0, 65271.0]])     # A-20 CO2: T [K], h [kJ/kmol]
# ---- route A: closed forms of the slide equations (SI: P kPa, R kJ/(kg K), V m/s, A m2, mdot kg/s)
def cp_of(k, R): return k * R / (k - 1.0)
def T0_of(T, V, cp): return T + V * V / (2000.0 * cp)                         # (1)/(7): V^2/2 in J/kg -> kJ/kg
def P0_of(P, T, T0, k): return P * (T0 / T) ** (k / (k - 1.0))               # (2)
def csound(k, R, T): return math.sqrt(k * R * 1000.0 * T)
def TT0(Ma, k): return 1.0 / (1.0 + (k - 1.0) / 2.0 * Ma * Ma)               # (9)
def PP0(Ma, k): return TT0(Ma, k) ** (k / (k - 1.0))                         # (10)
def AAs(Ma, k): return (1.0 / Ma) * ((2.0 / (k + 1.0)) * (1.0 + (k - 1.0) / 2.0 * Ma * Ma)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (17)
def crit(k): return dict(TT=2.0 / (k + 1.0), PP=(2.0 / (k + 1.0)) ** (k / (k - 1.0)))                                      # (12),(13)
def Ma_of_PP(pr, k): return math.sqrt(2.0 / (k - 1.0) * (pr ** (-(k - 1.0) / k) - 1.0))
def mdot_choked(A, P0, T0, k, R): return A * (P0 * 1000.0) * math.sqrt(k / (R * 1000.0 * T0)) * (2.0 / (k + 1.0)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (16) with P0 in Pa, R in J/(kg K)
# ---- route B: numerical descriptions that do not use (9)-(17)
def flux_max(k):
    """x* = T/T0 maximising the mass flux rho V ~ x^(1/(k-1)) sqrt(1-x) along the isentrope (energy equation V = sqrt(2 cp (T0 - T)))."""
    r = minimize_scalar(lambda x: -(x ** (1.0 / (k - 1.0)) * math.sqrt(1.0 - x)), bounds=(1e-6, 1 - 1e-9), method='bounded', options=dict(xatol=1e-13)); return r.x, -r.fun
def isentrope_T(T, P, Pnew, k):
    s = solve_ivp(lambda x, y: [(k - 1.0) / k], [0.0, math.log(Pnew / P)], [math.log(T)], rtol=1e-11, atol=1e-13); return math.exp(s.y[0, -1])
def c_num(T, k, R):
    """speed of sound as (dP/d rho)_s by central difference on the isentrope P = C rho^k (not sqrt(kRT))."""
    P0_ = 1.0e5; rho = P0_ / (R * 1000.0 * T); C = P0_ / rho ** k; h = 1e-5 * rho; return math.sqrt((C * (rho + h) ** k - C * (rho - h) ** k) / (2 * h))
def x_of_Ma(Ma, k, R=0.287):
    cp = cp_of(k, R); return brentq(lambda T: Ma * c_num(T, k, R) - math.sqrt(max(2000.0 * cp * (1000.0 - T), 0.0)), 1.0, 1000.0 - 1e-9, xtol=1e-12) / 1000.0
T01 = 300.15; pr = 9.0; cp, k, R, mdot = 1.005, 1.4, 0.287, 0.02
T02 = T01 * pr ** ((k - 1) / k); P_A = mdot * cp * (T02 - T01)
w_ds = abs(quad(lambda x: R * isentrope_T(T01, 1.0, math.exp(x), k), 0.0, math.log(pr))[0]); P_B = mdot * w_ds
h = lambda T: float(np.interp(T, A17[:, 0], A17[:, 1])); Pr = lambda T: float(np.interp(T, A17[:, 0], A17[:, 2]))
Pr2 = Pr(T01) * pr; T02v = float(np.interp(Pr2, A17[:, 2], A17[:, 0])); P_C = mdot * (h(T02v) - h(T01))
print(f"A closed form : T02 = {T02:.1f} K; power = {P_A:.3f} kW")
print(f"B integral    : power = {P_B:.3f} kW")
print(f"C Table A-17  : Pr(T01) = {Pr(T01):.3f} -> Pr2 = {Pr2:.3f} -> T02 = {T02v:.1f} K; power = {P_C:.3f} kW  (printed answer 5.27 kW)")
w16 = cp_of(1.33, 0.287) * 1023.15 * (1 - 0.1 ** (0.33 / 1.33)); print(f"slide p.16 (turbine, k = 1.33): w = {w16:.1f} kJ/kg")

A closed form : T02 = 562.3 K; power = 5.270 kW
B integral    : power = 5.267 kW
C Table A-17  : Pr(T01) = 1.388 -> Pr2 = 12.496 -> T02 = 558.0 K; power = 5.258 kW  (printed answer 5.27 kW)
slide p.16 (turbine, k = 1.33): w = 515.1 kJ/kg


**Đọc kết quả:** đường A cho 5,270 kW và đường B cho 5,267 kW (khác 0,05 % vì c_p); đường C (A-17) cho 5,258 kW: cả ba khớp đáp án in 5,27 kW trong 0,3 %. Phương pháp (A) vì thế đáng tin cho tr.14 và tr.16: bài tr.16 cho 515,1 kJ/kg (không có đáp án in).

## 3. Ống CO₂: slide tr.18–24 (Cengel Ví dụ 17–3)

**Câu hỏi:** CO₂ 3 kg/s, T₀ = 473 K, P₀ = 1400 kPa giãn nở đẳng entropi tới 200 kPa. Bảng T, V, ρ, c, A, Ma tại mỗi bước giảm 200 kPa tính bằng công thức và bằng tích phân số có khớp bảng slide tr.22 không?

In [3]:
import math, numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq, minimize_scalar
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A32 = np.array([[0.1, 0.1094, 5.8218, 0.993, 0.995, 0.998], [0.2, 0.2182, 2.9635, 0.9725, 0.9803, 0.9921], [0.3, 0.3257, 2.0351, 0.9395, 0.9564, 0.9823], [0.4, 0.4313, 1.5901, 0.8956, 0.9243, 0.969], [0.5, 0.5345, 1.3398, 0.843, 0.8852, 0.9524], [0.6, 0.6348, 1.1882, 0.784, 0.8405, 0.9328], [0.7, 0.7318, 1.0944, 0.7209, 0.7916, 0.9107], [0.8, 0.8251, 1.0382, 0.656, 0.74, 0.8865], [0.9, 0.9146, 1.0089, 0.5913, 0.687, 0.8606], [1.0, 1.0, 1.0, 0.5283, 0.6339, 0.8333], [1.2, 1.1583, 1.0304, 0.4124, 0.5311, 0.7764], [1.4, 1.2999, 1.1149, 0.3142, 0.4374, 0.7184], [1.6, 1.4254, 1.2502, 0.2353, 0.3557, 0.6614], [1.8, 1.536, 1.439, 0.174, 0.2868, 0.6068], [2.0, 1.633, 1.6875, 0.1278, 0.23, 0.5556], [2.2, 1.7179, 2.005, 0.0935, 0.1841, 0.5081], [2.4, 1.7922, 2.4031, 0.0684, 0.1472, 0.4647], [2.6, 1.8571, 2.896, 0.0501, 0.1179, 0.4252], [2.8, 1.914, 3.5001, 0.0368, 0.0946, 0.3894], [3.0, 1.964, 4.2346, 0.0272, 0.076, 0.3571], [5.0, 2.2361, 25.0, 0.0019, 0.0113, 0.1667]])     # A-32 (k = 1.4) rows: Ma, Ma*, A/A*, P/P0, rho/rho0, T/T0
A17 = np.array([[250.0, 250.05, 0.7329], [260.0, 260.09, 0.8405], [270.0, 270.11, 0.959], [280.0, 280.13, 1.0889], [285.0, 285.14, 1.1584], [290.0, 290.16, 1.2311], [295.0, 295.17, 1.3068], [298.0, 298.18, 1.3543], [300.0, 300.19, 1.386], [305.0, 305.22, 1.4686], [310.0, 310.24, 1.5546], [315.0, 315.27, 1.6442], [320.0, 320.29, 1.7375], [325.0, 325.31, 1.8345], [330.0, 330.34, 1.9352], [340.0, 340.42, 2.149], [350.0, 350.49, 2.379], [360.0, 360.58, 2.626], [370.0, 370.67, 2.892], [380.0, 380.77, 3.176], [390.0, 390.88, 3.481], [400.0, 400.98, 3.806], [410.0, 411.12, 4.153], [420.0, 421.26, 4.522], [430.0, 431.43, 4.915], [440.0, 441.61, 5.332], [450.0, 451.8, 5.775], [460.0, 462.02, 6.245], [470.0, 472.24, 6.742], [480.0, 482.49, 7.268], [490.0, 492.74, 7.824], [500.0, 503.02, 8.411], [510.0, 513.32, 9.031], [520.0, 523.63, 9.684], [530.0, 533.98, 10.37], [540.0, 544.35, 11.1], [550.0, 555.74, 11.86], [560.0, 565.17, 12.66], [570.0, 575.59, 13.5], [580.0, 586.04, 14.38], [590.0, 596.52, 15.31], [600.0, 607.02, 16.28], [610.0, 617.53, 17.3], [620.0, 628.07, 18.36], [630.0, 638.63, 19.84], [640.0, 649.22, 20.64], [650.0, 659.84, 21.86], [660.0, 670.47, 23.13], [670.0, 681.14, 24.46], [680.0, 691.82, 25.85], [690.0, 702.52, 27.29], [700.0, 713.27, 28.8], [710.0, 724.04, 30.38], [720.0, 734.82, 32.02], [730.0, 745.62, 33.72], [740.0, 756.44, 35.5], [750.0, 767.29, 37.35], [760.0, 778.18, 39.27], [780.0, 800.03, 43.35], [800.0, 821.95, 47.75]])     # A-17 air: T [K], h [kJ/kg], Pr
A20 = np.array([[300.0, 9431.0], [310.0, 9807.0], [320.0, 10186.0], [330.0, 10570.0], [340.0, 10959.0], [350.0, 11351.0], [360.0, 11748.0], [370.0, 12148.0], [380.0, 12552.0], [390.0, 12960.0], [400.0, 13372.0], [410.0, 13787.0], [420.0, 14206.0], [430.0, 14628.0], [440.0, 15054.0], [450.0, 15483.0], [460.0, 15916.0], [470.0, 16351.0], [480.0, 16791.0], [490.0, 17232.0], [500.0, 17678.0], [510.0, 18126.0], [520.0, 18576.0], [530.0, 19029.0], [540.0, 19485.0], [550.0, 19945.0], [560.0, 20407.0], [570.0, 20870.0], [580.0, 21337.0], [590.0, 21807.0], [600.0, 22280.0], [610.0, 22754.0], [620.0, 23231.0], [630.0, 23709.0], [640.0, 24190.0], [650.0, 24674.0], [660.0, 25160.0], [670.0, 25648.0], [680.0, 26138.0], [690.0, 26631.0], [700.0, 27125.0], [710.0, 27622.0], [720.0, 28121.0], [730.0, 28622.0], [740.0, 29124.0], [750.0, 29629.0], [760.0, 30135.0], [770.0, 30644.0], [780.0, 31154.0], [790.0, 31665.0], [800.0, 32179.0], [810.0, 32694.0], [820.0, 33212.0], [830.0, 33730.0], [840.0, 34251.0], [850.0, 34773.0], [860.0, 35296.0], [870.0, 35821.0], [880.0, 36347.0], [890.0, 36876.0], [900.0, 37405.0], [910.0, 37935.0], [920.0, 38467.0], [930.0, 39000.0], [940.0, 39535.0], [950.0, 40070.0], [960.0, 40607.0], [970.0, 41145.0], [980.0, 41685.0], [990.0, 42226.0], [1000.0, 42769.0], [1020.0, 43859.0], [1040.0, 44953.0], [1060.0, 46051.0], [1080.0, 47153.0], [1100.0, 48258.0], [1120.0, 49369.0], [1140.0, 50484.0], [1160.0, 51602.0], [1180.0, 52724.0], [1200.0, 53848.0], [1220.0, 54977.0], [1240.0, 56108.0], [1260.0, 57244.0], [1280.0, 58381.0], [1300.0, 59522.0], [1320.0, 60666.0], [1340.0, 61813.0], [1360.0, 62963.0], [1380.0, 64116.0], [1400.0, 65271.0]])     # A-20 CO2: T [K], h [kJ/kmol]
# ---- route A: closed forms of the slide equations (SI: P kPa, R kJ/(kg K), V m/s, A m2, mdot kg/s)
def cp_of(k, R): return k * R / (k - 1.0)
def T0_of(T, V, cp): return T + V * V / (2000.0 * cp)                         # (1)/(7): V^2/2 in J/kg -> kJ/kg
def P0_of(P, T, T0, k): return P * (T0 / T) ** (k / (k - 1.0))               # (2)
def csound(k, R, T): return math.sqrt(k * R * 1000.0 * T)
def TT0(Ma, k): return 1.0 / (1.0 + (k - 1.0) / 2.0 * Ma * Ma)               # (9)
def PP0(Ma, k): return TT0(Ma, k) ** (k / (k - 1.0))                         # (10)
def AAs(Ma, k): return (1.0 / Ma) * ((2.0 / (k + 1.0)) * (1.0 + (k - 1.0) / 2.0 * Ma * Ma)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (17)
def crit(k): return dict(TT=2.0 / (k + 1.0), PP=(2.0 / (k + 1.0)) ** (k / (k - 1.0)))                                      # (12),(13)
def Ma_of_PP(pr, k): return math.sqrt(2.0 / (k - 1.0) * (pr ** (-(k - 1.0) / k) - 1.0))
def mdot_choked(A, P0, T0, k, R): return A * (P0 * 1000.0) * math.sqrt(k / (R * 1000.0 * T0)) * (2.0 / (k + 1.0)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (16) with P0 in Pa, R in J/(kg K)
# ---- route B: numerical descriptions that do not use (9)-(17)
def flux_max(k):
    """x* = T/T0 maximising the mass flux rho V ~ x^(1/(k-1)) sqrt(1-x) along the isentrope (energy equation V = sqrt(2 cp (T0 - T)))."""
    r = minimize_scalar(lambda x: -(x ** (1.0 / (k - 1.0)) * math.sqrt(1.0 - x)), bounds=(1e-6, 1 - 1e-9), method='bounded', options=dict(xatol=1e-13)); return r.x, -r.fun
def isentrope_T(T, P, Pnew, k):
    s = solve_ivp(lambda x, y: [(k - 1.0) / k], [0.0, math.log(Pnew / P)], [math.log(T)], rtol=1e-11, atol=1e-13); return math.exp(s.y[0, -1])
def c_num(T, k, R):
    """speed of sound as (dP/d rho)_s by central difference on the isentrope P = C rho^k (not sqrt(kRT))."""
    P0_ = 1.0e5; rho = P0_ / (R * 1000.0 * T); C = P0_ / rho ** k; h = 1e-5 * rho; return math.sqrt((C * (rho + h) ** k - C * (rho - h) ** k) / (2 * h))
def x_of_Ma(Ma, k, R=0.287):
    cp = cp_of(k, R); return brentq(lambda T: Ma * c_num(T, k, R) - math.sqrt(max(2000.0 * cp * (1000.0 - T), 0.0)), 1.0, 1000.0 - 1e-9, xtol=1e-12) / 1000.0
k, R, T0, P0, mdot = 1.289, 0.1889, 473.0, 1400.0, 3.0
slide = {1200: (457, 164.5, 13.9, 333.6, 13.1, 0.493), 1000: (439, 240.7, 12.1, 326.9, 10.3, 0.736), 800: (417, 306.6, 10.1, 318.8, 9.64, 0.962), 767: (413, 317.2, 9.82, 317.2, 9.63, 1.000),
         600: (391, 371.4, 8.12, 308.7, 10.0, 1.203), 400: (357, 441.9, 5.93, 295.0, 11.5, 1.498), 200: (306, 530.9, 3.46, 272.9, 16.3, 1.946)}
cpkR = cp_of(k, R); print(f"cp = kR/(k-1) = {cpkR:.4f} (the slide quotes 0.846)")
print(" P     T      V      rho     c      A(cm2)  Ma     | slide T, V, A, Ma")
for P in (1200, 1000, 800, 767, 600, 400, 200):
    T = T0 * (P / P0) ** ((k - 1) / k); V = math.sqrt(2000.0 * cpkR * (T0 - T)); rho = P / (R * T); c = csound(k, R, T); A = mdot / (rho * V) * 1e4
    Tb = isentrope_T(T0, P0, float(P), k); Vb = math.sqrt(2000.0 * quad(lambda t: cpkR, Tb, T0)[0]); cb = c_num(Tb, k, R)           # route B
    assert abs(Tb - T) < 1e-6 and abs(Vb / V - 1) < 1e-9 and abs(cb / c - 1) < 1e-6
    s = slide[P]; print(f"{P:5d} {T:6.1f} {V:7.1f} {rho:7.2f} {c:6.1f} {A:7.2f} {V / c:6.3f} | {s[0]} {s[1]} {s[4]} {s[5]}")
print("slide with cp = 0.846 at 1200 kPa: V =", round(math.sqrt(2000 * 0.846 * (T0 - T0 * (1200 / P0) ** ((k - 1) / k))), 1), "m/s")

cp = kR/(k-1) = 0.8425 (the slide quotes 0.846)
 P     T      V      rho     c      A(cm2)  Ma     | slide T, V, A, Ma
 1200  456.9   164.5   13.90  333.6   13.11  0.493 | 457 164.5 13.1 0.493
 1000  438.6   240.7   12.07  326.8   10.33  0.736 | 439 240.7 10.3 0.736
  800  417.2   306.6   10.15  318.7    9.64  0.962 | 417 306.6 9.64 0.962
  767  413.3   317.2    9.82  317.2    9.63  1.000 | 413 317.2 9.63 1.0
  600  391.2   371.3    8.12  308.6    9.95  1.203 | 391 371.4 10.0 1.203
  400  357.2   441.8    5.93  294.9   11.45  1.498 | 357 441.9 11.5 1.498
  200  305.8   530.9    3.46  272.9   16.32  1.946 | 306 530.9 16.3 1.946
slide with cp = 0.846 at 1200 kPa: V = 164.9 m/s


**Đọc kết quả:** tại 1200 kPa: T = 456,9 K, V = 164,5 m/s, ρ = 13,90 kg/m³, A = 13,1 cm², Ma = 0,493; tại cổ họng 767 kPa: A = 9,63 cm² (cực tiểu) và Ma = 1,000. Bảng slide khớp c_p = kR/(k−1) = 0,8425 (V lệch ≤ 0,3 % so với 0,846), nên đã dùng 0,8425; c_p = 0,846 cho V = 164,9 m/s ở 1200 kPa (slide in 164,5). Đường B trùng đường A (khẳng định bằng assert).

## 4. Tính chất tới hạn: cực đại số của lưu lượng ρV so với công thức (12)–(14)

**Câu hỏi:** công thức (13) P*/P₀ = (2/(k+1))^(k/(k−1)) có đúng là vị trí cực đại của khối lượng riêng nhân vận tốc ρV (lưu lượng trên đơn vị diện tích) dọc đường đẳng entropi không? Thử cho không khí, CO₂ (slide tr.36–38) và sản phẩm cháy (tr.16).

In [4]:
import math, numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq, minimize_scalar
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A32 = np.array([[0.1, 0.1094, 5.8218, 0.993, 0.995, 0.998], [0.2, 0.2182, 2.9635, 0.9725, 0.9803, 0.9921], [0.3, 0.3257, 2.0351, 0.9395, 0.9564, 0.9823], [0.4, 0.4313, 1.5901, 0.8956, 0.9243, 0.969], [0.5, 0.5345, 1.3398, 0.843, 0.8852, 0.9524], [0.6, 0.6348, 1.1882, 0.784, 0.8405, 0.9328], [0.7, 0.7318, 1.0944, 0.7209, 0.7916, 0.9107], [0.8, 0.8251, 1.0382, 0.656, 0.74, 0.8865], [0.9, 0.9146, 1.0089, 0.5913, 0.687, 0.8606], [1.0, 1.0, 1.0, 0.5283, 0.6339, 0.8333], [1.2, 1.1583, 1.0304, 0.4124, 0.5311, 0.7764], [1.4, 1.2999, 1.1149, 0.3142, 0.4374, 0.7184], [1.6, 1.4254, 1.2502, 0.2353, 0.3557, 0.6614], [1.8, 1.536, 1.439, 0.174, 0.2868, 0.6068], [2.0, 1.633, 1.6875, 0.1278, 0.23, 0.5556], [2.2, 1.7179, 2.005, 0.0935, 0.1841, 0.5081], [2.4, 1.7922, 2.4031, 0.0684, 0.1472, 0.4647], [2.6, 1.8571, 2.896, 0.0501, 0.1179, 0.4252], [2.8, 1.914, 3.5001, 0.0368, 0.0946, 0.3894], [3.0, 1.964, 4.2346, 0.0272, 0.076, 0.3571], [5.0, 2.2361, 25.0, 0.0019, 0.0113, 0.1667]])     # A-32 (k = 1.4) rows: Ma, Ma*, A/A*, P/P0, rho/rho0, T/T0
A17 = np.array([[250.0, 250.05, 0.7329], [260.0, 260.09, 0.8405], [270.0, 270.11, 0.959], [280.0, 280.13, 1.0889], [285.0, 285.14, 1.1584], [290.0, 290.16, 1.2311], [295.0, 295.17, 1.3068], [298.0, 298.18, 1.3543], [300.0, 300.19, 1.386], [305.0, 305.22, 1.4686], [310.0, 310.24, 1.5546], [315.0, 315.27, 1.6442], [320.0, 320.29, 1.7375], [325.0, 325.31, 1.8345], [330.0, 330.34, 1.9352], [340.0, 340.42, 2.149], [350.0, 350.49, 2.379], [360.0, 360.58, 2.626], [370.0, 370.67, 2.892], [380.0, 380.77, 3.176], [390.0, 390.88, 3.481], [400.0, 400.98, 3.806], [410.0, 411.12, 4.153], [420.0, 421.26, 4.522], [430.0, 431.43, 4.915], [440.0, 441.61, 5.332], [450.0, 451.8, 5.775], [460.0, 462.02, 6.245], [470.0, 472.24, 6.742], [480.0, 482.49, 7.268], [490.0, 492.74, 7.824], [500.0, 503.02, 8.411], [510.0, 513.32, 9.031], [520.0, 523.63, 9.684], [530.0, 533.98, 10.37], [540.0, 544.35, 11.1], [550.0, 555.74, 11.86], [560.0, 565.17, 12.66], [570.0, 575.59, 13.5], [580.0, 586.04, 14.38], [590.0, 596.52, 15.31], [600.0, 607.02, 16.28], [610.0, 617.53, 17.3], [620.0, 628.07, 18.36], [630.0, 638.63, 19.84], [640.0, 649.22, 20.64], [650.0, 659.84, 21.86], [660.0, 670.47, 23.13], [670.0, 681.14, 24.46], [680.0, 691.82, 25.85], [690.0, 702.52, 27.29], [700.0, 713.27, 28.8], [710.0, 724.04, 30.38], [720.0, 734.82, 32.02], [730.0, 745.62, 33.72], [740.0, 756.44, 35.5], [750.0, 767.29, 37.35], [760.0, 778.18, 39.27], [780.0, 800.03, 43.35], [800.0, 821.95, 47.75]])     # A-17 air: T [K], h [kJ/kg], Pr
A20 = np.array([[300.0, 9431.0], [310.0, 9807.0], [320.0, 10186.0], [330.0, 10570.0], [340.0, 10959.0], [350.0, 11351.0], [360.0, 11748.0], [370.0, 12148.0], [380.0, 12552.0], [390.0, 12960.0], [400.0, 13372.0], [410.0, 13787.0], [420.0, 14206.0], [430.0, 14628.0], [440.0, 15054.0], [450.0, 15483.0], [460.0, 15916.0], [470.0, 16351.0], [480.0, 16791.0], [490.0, 17232.0], [500.0, 17678.0], [510.0, 18126.0], [520.0, 18576.0], [530.0, 19029.0], [540.0, 19485.0], [550.0, 19945.0], [560.0, 20407.0], [570.0, 20870.0], [580.0, 21337.0], [590.0, 21807.0], [600.0, 22280.0], [610.0, 22754.0], [620.0, 23231.0], [630.0, 23709.0], [640.0, 24190.0], [650.0, 24674.0], [660.0, 25160.0], [670.0, 25648.0], [680.0, 26138.0], [690.0, 26631.0], [700.0, 27125.0], [710.0, 27622.0], [720.0, 28121.0], [730.0, 28622.0], [740.0, 29124.0], [750.0, 29629.0], [760.0, 30135.0], [770.0, 30644.0], [780.0, 31154.0], [790.0, 31665.0], [800.0, 32179.0], [810.0, 32694.0], [820.0, 33212.0], [830.0, 33730.0], [840.0, 34251.0], [850.0, 34773.0], [860.0, 35296.0], [870.0, 35821.0], [880.0, 36347.0], [890.0, 36876.0], [900.0, 37405.0], [910.0, 37935.0], [920.0, 38467.0], [930.0, 39000.0], [940.0, 39535.0], [950.0, 40070.0], [960.0, 40607.0], [970.0, 41145.0], [980.0, 41685.0], [990.0, 42226.0], [1000.0, 42769.0], [1020.0, 43859.0], [1040.0, 44953.0], [1060.0, 46051.0], [1080.0, 47153.0], [1100.0, 48258.0], [1120.0, 49369.0], [1140.0, 50484.0], [1160.0, 51602.0], [1180.0, 52724.0], [1200.0, 53848.0], [1220.0, 54977.0], [1240.0, 56108.0], [1260.0, 57244.0], [1280.0, 58381.0], [1300.0, 59522.0], [1320.0, 60666.0], [1340.0, 61813.0], [1360.0, 62963.0], [1380.0, 64116.0], [1400.0, 65271.0]])     # A-20 CO2: T [K], h [kJ/kmol]
# ---- route A: closed forms of the slide equations (SI: P kPa, R kJ/(kg K), V m/s, A m2, mdot kg/s)
def cp_of(k, R): return k * R / (k - 1.0)
def T0_of(T, V, cp): return T + V * V / (2000.0 * cp)                         # (1)/(7): V^2/2 in J/kg -> kJ/kg
def P0_of(P, T, T0, k): return P * (T0 / T) ** (k / (k - 1.0))               # (2)
def csound(k, R, T): return math.sqrt(k * R * 1000.0 * T)
def TT0(Ma, k): return 1.0 / (1.0 + (k - 1.0) / 2.0 * Ma * Ma)               # (9)
def PP0(Ma, k): return TT0(Ma, k) ** (k / (k - 1.0))                         # (10)
def AAs(Ma, k): return (1.0 / Ma) * ((2.0 / (k + 1.0)) * (1.0 + (k - 1.0) / 2.0 * Ma * Ma)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (17)
def crit(k): return dict(TT=2.0 / (k + 1.0), PP=(2.0 / (k + 1.0)) ** (k / (k - 1.0)))                                      # (12),(13)
def Ma_of_PP(pr, k): return math.sqrt(2.0 / (k - 1.0) * (pr ** (-(k - 1.0) / k) - 1.0))
def mdot_choked(A, P0, T0, k, R): return A * (P0 * 1000.0) * math.sqrt(k / (R * 1000.0 * T0)) * (2.0 / (k + 1.0)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (16) with P0 in Pa, R in J/(kg K)
# ---- route B: numerical descriptions that do not use (9)-(17)
def flux_max(k):
    """x* = T/T0 maximising the mass flux rho V ~ x^(1/(k-1)) sqrt(1-x) along the isentrope (energy equation V = sqrt(2 cp (T0 - T)))."""
    r = minimize_scalar(lambda x: -(x ** (1.0 / (k - 1.0)) * math.sqrt(1.0 - x)), bounds=(1e-6, 1 - 1e-9), method='bounded', options=dict(xatol=1e-13)); return r.x, -r.fun
def isentrope_T(T, P, Pnew, k):
    s = solve_ivp(lambda x, y: [(k - 1.0) / k], [0.0, math.log(Pnew / P)], [math.log(T)], rtol=1e-11, atol=1e-13); return math.exp(s.y[0, -1])
def c_num(T, k, R):
    """speed of sound as (dP/d rho)_s by central difference on the isentrope P = C rho^k (not sqrt(kRT))."""
    P0_ = 1.0e5; rho = P0_ / (R * 1000.0 * T); C = P0_ / rho ** k; h = 1e-5 * rho; return math.sqrt((C * (rho + h) ** k - C * (rho - h) ** k) / (2 * h))
def x_of_Ma(Ma, k, R=0.287):
    cp = cp_of(k, R); return brentq(lambda T: Ma * c_num(T, k, R) - math.sqrt(max(2000.0 * cp * (1000.0 - T), 0.0)), 1.0, 1000.0 - 1e-9, xtol=1e-12) / 1000.0
for name, k in (('air', 1.4), ('CO2 (slide p.37)', 1.289), ('products k = 1.33', 1.33), ('helium', 1.667)):
    xs, fs = flux_max(k); c = crit(k); Pstar = math.exp(k / (k - 1.0) * math.log(xs))
    print(f"{name:20s}: numerical T*/T0 = {xs:.5f} (eq.12 {c['TT']:.5f}); numerical P*/P0 = {Pstar:.5f} (eq.13 {c['PP']:.5f}); Ma at the maximum = {math.sqrt(2/(k-1)*(1/xs-1)):.5f}")
print("slide Ex 17-4: T* =", round(crit(1.289)['TT'] * 473.0, 1), "K; P* =", round(crit(1.289)['PP'] * 1400.0, 1), "kPa")

air                 : numerical T*/T0 = 0.83333 (eq.12 0.83333); numerical P*/P0 = 0.52828 (eq.13 0.52828); Ma at the maximum = 1.00000
CO2 (slide p.37)    : numerical T*/T0 = 0.87374 (eq.12 0.87374); numerical P*/P0 = 0.54772 (eq.13 0.54772); Ma at the maximum = 1.00000
products k = 1.33   : numerical T*/T0 = 0.85837 (eq.12 0.85837); numerical P*/P0 = 0.54036 (eq.13 0.54036); Ma at the maximum = 1.00000
helium              : numerical T*/T0 = 0.74991 (eq.12 0.74991); numerical P*/P0 = 0.48709 (eq.13 0.48709); Ma at the maximum = 1.00000
slide Ex 17-4: T* = 413.3 K; P* = 766.8 kPa


**Đọc kết quả:** cực đại số của ρV rơi đúng tại T*/T₀ = 2/(k+1) và P*/P₀ của (13) cho cả bốn loại khí, và Ma tại cực đại bằng 1,00000: tức Ma = 1 chính là điều kiện làm lưu lượng cực đại (slide tr.45). Với CO₂: T* = 413,3 K, P* = 766,8 kPa (slide 413 K; 767 kPa).

## 5. Vòi hội tụ và áp suất ngược: slide tr.50–55 (Cengel Ví dụ 17–5) và bẫy √1000

**Câu hỏi:** không khí 1 MPa, 600 °C, 150 m/s vào vòi hội tụ A_t = 50 cm². Lưu lượng khối với P_b = 0,7 MPa và 0,4 MPa là bao nhiêu? Tính bằng (15)/(16) và bằng năng lượng + đẳng entropi (không dùng (15), (16)); thử công thức (16) nguyên văn bằng kPa và kJ/(kg·K) như slide tr.55.

In [5]:
import math, numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq, minimize_scalar
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A32 = np.array([[0.1, 0.1094, 5.8218, 0.993, 0.995, 0.998], [0.2, 0.2182, 2.9635, 0.9725, 0.9803, 0.9921], [0.3, 0.3257, 2.0351, 0.9395, 0.9564, 0.9823], [0.4, 0.4313, 1.5901, 0.8956, 0.9243, 0.969], [0.5, 0.5345, 1.3398, 0.843, 0.8852, 0.9524], [0.6, 0.6348, 1.1882, 0.784, 0.8405, 0.9328], [0.7, 0.7318, 1.0944, 0.7209, 0.7916, 0.9107], [0.8, 0.8251, 1.0382, 0.656, 0.74, 0.8865], [0.9, 0.9146, 1.0089, 0.5913, 0.687, 0.8606], [1.0, 1.0, 1.0, 0.5283, 0.6339, 0.8333], [1.2, 1.1583, 1.0304, 0.4124, 0.5311, 0.7764], [1.4, 1.2999, 1.1149, 0.3142, 0.4374, 0.7184], [1.6, 1.4254, 1.2502, 0.2353, 0.3557, 0.6614], [1.8, 1.536, 1.439, 0.174, 0.2868, 0.6068], [2.0, 1.633, 1.6875, 0.1278, 0.23, 0.5556], [2.2, 1.7179, 2.005, 0.0935, 0.1841, 0.5081], [2.4, 1.7922, 2.4031, 0.0684, 0.1472, 0.4647], [2.6, 1.8571, 2.896, 0.0501, 0.1179, 0.4252], [2.8, 1.914, 3.5001, 0.0368, 0.0946, 0.3894], [3.0, 1.964, 4.2346, 0.0272, 0.076, 0.3571], [5.0, 2.2361, 25.0, 0.0019, 0.0113, 0.1667]])     # A-32 (k = 1.4) rows: Ma, Ma*, A/A*, P/P0, rho/rho0, T/T0
A17 = np.array([[250.0, 250.05, 0.7329], [260.0, 260.09, 0.8405], [270.0, 270.11, 0.959], [280.0, 280.13, 1.0889], [285.0, 285.14, 1.1584], [290.0, 290.16, 1.2311], [295.0, 295.17, 1.3068], [298.0, 298.18, 1.3543], [300.0, 300.19, 1.386], [305.0, 305.22, 1.4686], [310.0, 310.24, 1.5546], [315.0, 315.27, 1.6442], [320.0, 320.29, 1.7375], [325.0, 325.31, 1.8345], [330.0, 330.34, 1.9352], [340.0, 340.42, 2.149], [350.0, 350.49, 2.379], [360.0, 360.58, 2.626], [370.0, 370.67, 2.892], [380.0, 380.77, 3.176], [390.0, 390.88, 3.481], [400.0, 400.98, 3.806], [410.0, 411.12, 4.153], [420.0, 421.26, 4.522], [430.0, 431.43, 4.915], [440.0, 441.61, 5.332], [450.0, 451.8, 5.775], [460.0, 462.02, 6.245], [470.0, 472.24, 6.742], [480.0, 482.49, 7.268], [490.0, 492.74, 7.824], [500.0, 503.02, 8.411], [510.0, 513.32, 9.031], [520.0, 523.63, 9.684], [530.0, 533.98, 10.37], [540.0, 544.35, 11.1], [550.0, 555.74, 11.86], [560.0, 565.17, 12.66], [570.0, 575.59, 13.5], [580.0, 586.04, 14.38], [590.0, 596.52, 15.31], [600.0, 607.02, 16.28], [610.0, 617.53, 17.3], [620.0, 628.07, 18.36], [630.0, 638.63, 19.84], [640.0, 649.22, 20.64], [650.0, 659.84, 21.86], [660.0, 670.47, 23.13], [670.0, 681.14, 24.46], [680.0, 691.82, 25.85], [690.0, 702.52, 27.29], [700.0, 713.27, 28.8], [710.0, 724.04, 30.38], [720.0, 734.82, 32.02], [730.0, 745.62, 33.72], [740.0, 756.44, 35.5], [750.0, 767.29, 37.35], [760.0, 778.18, 39.27], [780.0, 800.03, 43.35], [800.0, 821.95, 47.75]])     # A-17 air: T [K], h [kJ/kg], Pr
A20 = np.array([[300.0, 9431.0], [310.0, 9807.0], [320.0, 10186.0], [330.0, 10570.0], [340.0, 10959.0], [350.0, 11351.0], [360.0, 11748.0], [370.0, 12148.0], [380.0, 12552.0], [390.0, 12960.0], [400.0, 13372.0], [410.0, 13787.0], [420.0, 14206.0], [430.0, 14628.0], [440.0, 15054.0], [450.0, 15483.0], [460.0, 15916.0], [470.0, 16351.0], [480.0, 16791.0], [490.0, 17232.0], [500.0, 17678.0], [510.0, 18126.0], [520.0, 18576.0], [530.0, 19029.0], [540.0, 19485.0], [550.0, 19945.0], [560.0, 20407.0], [570.0, 20870.0], [580.0, 21337.0], [590.0, 21807.0], [600.0, 22280.0], [610.0, 22754.0], [620.0, 23231.0], [630.0, 23709.0], [640.0, 24190.0], [650.0, 24674.0], [660.0, 25160.0], [670.0, 25648.0], [680.0, 26138.0], [690.0, 26631.0], [700.0, 27125.0], [710.0, 27622.0], [720.0, 28121.0], [730.0, 28622.0], [740.0, 29124.0], [750.0, 29629.0], [760.0, 30135.0], [770.0, 30644.0], [780.0, 31154.0], [790.0, 31665.0], [800.0, 32179.0], [810.0, 32694.0], [820.0, 33212.0], [830.0, 33730.0], [840.0, 34251.0], [850.0, 34773.0], [860.0, 35296.0], [870.0, 35821.0], [880.0, 36347.0], [890.0, 36876.0], [900.0, 37405.0], [910.0, 37935.0], [920.0, 38467.0], [930.0, 39000.0], [940.0, 39535.0], [950.0, 40070.0], [960.0, 40607.0], [970.0, 41145.0], [980.0, 41685.0], [990.0, 42226.0], [1000.0, 42769.0], [1020.0, 43859.0], [1040.0, 44953.0], [1060.0, 46051.0], [1080.0, 47153.0], [1100.0, 48258.0], [1120.0, 49369.0], [1140.0, 50484.0], [1160.0, 51602.0], [1180.0, 52724.0], [1200.0, 53848.0], [1220.0, 54977.0], [1240.0, 56108.0], [1260.0, 57244.0], [1280.0, 58381.0], [1300.0, 59522.0], [1320.0, 60666.0], [1340.0, 61813.0], [1360.0, 62963.0], [1380.0, 64116.0], [1400.0, 65271.0]])     # A-20 CO2: T [K], h [kJ/kmol]
# ---- route A: closed forms of the slide equations (SI: P kPa, R kJ/(kg K), V m/s, A m2, mdot kg/s)
def cp_of(k, R): return k * R / (k - 1.0)
def T0_of(T, V, cp): return T + V * V / (2000.0 * cp)                         # (1)/(7): V^2/2 in J/kg -> kJ/kg
def P0_of(P, T, T0, k): return P * (T0 / T) ** (k / (k - 1.0))               # (2)
def csound(k, R, T): return math.sqrt(k * R * 1000.0 * T)
def TT0(Ma, k): return 1.0 / (1.0 + (k - 1.0) / 2.0 * Ma * Ma)               # (9)
def PP0(Ma, k): return TT0(Ma, k) ** (k / (k - 1.0))                         # (10)
def AAs(Ma, k): return (1.0 / Ma) * ((2.0 / (k + 1.0)) * (1.0 + (k - 1.0) / 2.0 * Ma * Ma)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (17)
def crit(k): return dict(TT=2.0 / (k + 1.0), PP=(2.0 / (k + 1.0)) ** (k / (k - 1.0)))                                      # (12),(13)
def Ma_of_PP(pr, k): return math.sqrt(2.0 / (k - 1.0) * (pr ** (-(k - 1.0) / k) - 1.0))
def mdot_choked(A, P0, T0, k, R): return A * (P0 * 1000.0) * math.sqrt(k / (R * 1000.0 * T0)) * (2.0 / (k + 1.0)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (16) with P0 in Pa, R in J/(kg K)
# ---- route B: numerical descriptions that do not use (9)-(17)
def flux_max(k):
    """x* = T/T0 maximising the mass flux rho V ~ x^(1/(k-1)) sqrt(1-x) along the isentrope (energy equation V = sqrt(2 cp (T0 - T)))."""
    r = minimize_scalar(lambda x: -(x ** (1.0 / (k - 1.0)) * math.sqrt(1.0 - x)), bounds=(1e-6, 1 - 1e-9), method='bounded', options=dict(xatol=1e-13)); return r.x, -r.fun
def isentrope_T(T, P, Pnew, k):
    s = solve_ivp(lambda x, y: [(k - 1.0) / k], [0.0, math.log(Pnew / P)], [math.log(T)], rtol=1e-11, atol=1e-13); return math.exp(s.y[0, -1])
def c_num(T, k, R):
    """speed of sound as (dP/d rho)_s by central difference on the isentrope P = C rho^k (not sqrt(kRT))."""
    P0_ = 1.0e5; rho = P0_ / (R * 1000.0 * T); C = P0_ / rho ** k; h = 1e-5 * rho; return math.sqrt((C * (rho + h) ** k - C * (rho - h) ** k) / (2 * h))
def x_of_Ma(Ma, k, R=0.287):
    cp = cp_of(k, R); return brentq(lambda T: Ma * c_num(T, k, R) - math.sqrt(max(2000.0 * cp * (1000.0 - T), 0.0)), 1.0, 1000.0 - 1e-9, xtol=1e-12) / 1000.0
k, R, cp, A = 1.4, 0.287, 1.005, 50e-4
T0 = 873.0 + 150.0 ** 2 / (2000.0 * cp); P0 = 1000.0 * (T0 / 873.0) ** 3.5; Pc = P0 * crit(k)['PP']
print(f"T0 = {T0:.1f} K; P0 = {P0:.1f} kPa; P* = {Pc:.1f} kPa")
xs, fs = flux_max(k)
for Pb in (700.0, 400.0):
    Pe = max(Pb, P0 * math.exp(k / (k - 1) * math.log(xs)))                                  # route B: exit pressure = max(Pb, pressure at flux maximum)
    Te = T0 * (Pe / P0) ** ((k - 1) / k); V = math.sqrt(2000.0 * cp_of(k, R) * (T0 - Te)); mB = Pe / (R * Te) * V * A
    Ma = Ma_of_PP(min(Pe, P0) / P0, k); mA = mdot_choked(A, P0, T0, k, R) if Pb < Pc else (Pe / (R * T0 * TT0(Ma, k))) * A * Ma * csound(k, R, T0 * TT0(Ma, k))
    print(f"Pb = {Pb:5.0f} kPa: {'choked' if Pb < Pc else 'not choked'}; Pe = {Pe:6.1f} kPa; Ma_e = {V / csound(k, R, Te):.4f}; V = {V:6.1f} m/s; mdot A = {mA:.3f}, B = {mB:.3f} kg/s")
lit = A * 1045.0 * math.sqrt(1.4 / (0.287 * 884.0)) * (2 / 2.4) ** (2.4 / 0.8); ok = mdot_choked(A, 1045.0, 884.0, k, R)
print(f"slide p.55 literally (kPa, kJ/(kg K)): {lit:.4f} kg/s ; with P0 in Pa and R in J/(kg K): {ok:.3f} kg/s ; ratio = {ok / lit:.2f} = sqrt(1000) = {math.sqrt(1000):.2f}")

T0 = 884.2 K; P0 = 1045.6 kPa; P* = 552.4 kPa
Pb =   700 kPa: not choked; Pe =  700.0 kPa; Ma_e = 0.7794; V =  438.7 m/s; mdot A = 6.785, B = 6.785 kg/s
Pb =   400 kPa: choked; Pe =  552.4 kPa; Ma_e = 1.0000; V =  544.1 m/s; mdot A = 7.106, B = 7.106 kg/s
slide p.55 literally (kPa, kJ/(kg K)): 0.2246 kg/s ; with P0 in Pa and R in J/(kg K): 7.103 kg/s ; ratio = 31.62 = sqrt(1000) = 31.62


**Đọc kết quả:** T₀ = 884,2 K, P₀ = 1045,6 kPa, P* = 552,4 kPa. P_b = 700 kPa: không nghẹt, Ma_e = 0,7794, ṁ = 6,785 kg/s (slide 6,77 từ nội suy A-32). P_b = 400 kPa: nghẹt, P_e = P* = 552,4 kPa, ṁ = 7,106 kg/s (slide 7,10). Hai đường A, B trùng. Thay nguyên văn kPa và kJ/(kg·K) cho 0,2246 kg/s (sai); đổi đúng đơn vị cho 7,103 kg/s: tỉ số chính là √1000 = 31,62. Đây là lỗi thiếu hệ số đổi đơn vị của slide tr.55.

## 6. Tỉ số diện tích A/A*: slide tr.56–59 (Cengel Ví dụ 17–6) và nội suy bảng A-32

**Câu hỏi:** N₂ (k = 1,4) ở Ma₁ = 0,3, T₁ = 400 K, P₁ = 100 kPa; diện tích giảm 20 %. Ma₂, T₂, P₂ chính xác là bao nhiêu và nội suy tuyến tính của slide sai bao nhiêu? Ba đường: brentq trên (17), tích phân ODE Hugoniot, nội suy A-32.

In [6]:
import math, numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq, minimize_scalar
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A32 = np.array([[0.1, 0.1094, 5.8218, 0.993, 0.995, 0.998], [0.2, 0.2182, 2.9635, 0.9725, 0.9803, 0.9921], [0.3, 0.3257, 2.0351, 0.9395, 0.9564, 0.9823], [0.4, 0.4313, 1.5901, 0.8956, 0.9243, 0.969], [0.5, 0.5345, 1.3398, 0.843, 0.8852, 0.9524], [0.6, 0.6348, 1.1882, 0.784, 0.8405, 0.9328], [0.7, 0.7318, 1.0944, 0.7209, 0.7916, 0.9107], [0.8, 0.8251, 1.0382, 0.656, 0.74, 0.8865], [0.9, 0.9146, 1.0089, 0.5913, 0.687, 0.8606], [1.0, 1.0, 1.0, 0.5283, 0.6339, 0.8333], [1.2, 1.1583, 1.0304, 0.4124, 0.5311, 0.7764], [1.4, 1.2999, 1.1149, 0.3142, 0.4374, 0.7184], [1.6, 1.4254, 1.2502, 0.2353, 0.3557, 0.6614], [1.8, 1.536, 1.439, 0.174, 0.2868, 0.6068], [2.0, 1.633, 1.6875, 0.1278, 0.23, 0.5556], [2.2, 1.7179, 2.005, 0.0935, 0.1841, 0.5081], [2.4, 1.7922, 2.4031, 0.0684, 0.1472, 0.4647], [2.6, 1.8571, 2.896, 0.0501, 0.1179, 0.4252], [2.8, 1.914, 3.5001, 0.0368, 0.0946, 0.3894], [3.0, 1.964, 4.2346, 0.0272, 0.076, 0.3571], [5.0, 2.2361, 25.0, 0.0019, 0.0113, 0.1667]])     # A-32 (k = 1.4) rows: Ma, Ma*, A/A*, P/P0, rho/rho0, T/T0
A17 = np.array([[250.0, 250.05, 0.7329], [260.0, 260.09, 0.8405], [270.0, 270.11, 0.959], [280.0, 280.13, 1.0889], [285.0, 285.14, 1.1584], [290.0, 290.16, 1.2311], [295.0, 295.17, 1.3068], [298.0, 298.18, 1.3543], [300.0, 300.19, 1.386], [305.0, 305.22, 1.4686], [310.0, 310.24, 1.5546], [315.0, 315.27, 1.6442], [320.0, 320.29, 1.7375], [325.0, 325.31, 1.8345], [330.0, 330.34, 1.9352], [340.0, 340.42, 2.149], [350.0, 350.49, 2.379], [360.0, 360.58, 2.626], [370.0, 370.67, 2.892], [380.0, 380.77, 3.176], [390.0, 390.88, 3.481], [400.0, 400.98, 3.806], [410.0, 411.12, 4.153], [420.0, 421.26, 4.522], [430.0, 431.43, 4.915], [440.0, 441.61, 5.332], [450.0, 451.8, 5.775], [460.0, 462.02, 6.245], [470.0, 472.24, 6.742], [480.0, 482.49, 7.268], [490.0, 492.74, 7.824], [500.0, 503.02, 8.411], [510.0, 513.32, 9.031], [520.0, 523.63, 9.684], [530.0, 533.98, 10.37], [540.0, 544.35, 11.1], [550.0, 555.74, 11.86], [560.0, 565.17, 12.66], [570.0, 575.59, 13.5], [580.0, 586.04, 14.38], [590.0, 596.52, 15.31], [600.0, 607.02, 16.28], [610.0, 617.53, 17.3], [620.0, 628.07, 18.36], [630.0, 638.63, 19.84], [640.0, 649.22, 20.64], [650.0, 659.84, 21.86], [660.0, 670.47, 23.13], [670.0, 681.14, 24.46], [680.0, 691.82, 25.85], [690.0, 702.52, 27.29], [700.0, 713.27, 28.8], [710.0, 724.04, 30.38], [720.0, 734.82, 32.02], [730.0, 745.62, 33.72], [740.0, 756.44, 35.5], [750.0, 767.29, 37.35], [760.0, 778.18, 39.27], [780.0, 800.03, 43.35], [800.0, 821.95, 47.75]])     # A-17 air: T [K], h [kJ/kg], Pr
A20 = np.array([[300.0, 9431.0], [310.0, 9807.0], [320.0, 10186.0], [330.0, 10570.0], [340.0, 10959.0], [350.0, 11351.0], [360.0, 11748.0], [370.0, 12148.0], [380.0, 12552.0], [390.0, 12960.0], [400.0, 13372.0], [410.0, 13787.0], [420.0, 14206.0], [430.0, 14628.0], [440.0, 15054.0], [450.0, 15483.0], [460.0, 15916.0], [470.0, 16351.0], [480.0, 16791.0], [490.0, 17232.0], [500.0, 17678.0], [510.0, 18126.0], [520.0, 18576.0], [530.0, 19029.0], [540.0, 19485.0], [550.0, 19945.0], [560.0, 20407.0], [570.0, 20870.0], [580.0, 21337.0], [590.0, 21807.0], [600.0, 22280.0], [610.0, 22754.0], [620.0, 23231.0], [630.0, 23709.0], [640.0, 24190.0], [650.0, 24674.0], [660.0, 25160.0], [670.0, 25648.0], [680.0, 26138.0], [690.0, 26631.0], [700.0, 27125.0], [710.0, 27622.0], [720.0, 28121.0], [730.0, 28622.0], [740.0, 29124.0], [750.0, 29629.0], [760.0, 30135.0], [770.0, 30644.0], [780.0, 31154.0], [790.0, 31665.0], [800.0, 32179.0], [810.0, 32694.0], [820.0, 33212.0], [830.0, 33730.0], [840.0, 34251.0], [850.0, 34773.0], [860.0, 35296.0], [870.0, 35821.0], [880.0, 36347.0], [890.0, 36876.0], [900.0, 37405.0], [910.0, 37935.0], [920.0, 38467.0], [930.0, 39000.0], [940.0, 39535.0], [950.0, 40070.0], [960.0, 40607.0], [970.0, 41145.0], [980.0, 41685.0], [990.0, 42226.0], [1000.0, 42769.0], [1020.0, 43859.0], [1040.0, 44953.0], [1060.0, 46051.0], [1080.0, 47153.0], [1100.0, 48258.0], [1120.0, 49369.0], [1140.0, 50484.0], [1160.0, 51602.0], [1180.0, 52724.0], [1200.0, 53848.0], [1220.0, 54977.0], [1240.0, 56108.0], [1260.0, 57244.0], [1280.0, 58381.0], [1300.0, 59522.0], [1320.0, 60666.0], [1340.0, 61813.0], [1360.0, 62963.0], [1380.0, 64116.0], [1400.0, 65271.0]])     # A-20 CO2: T [K], h [kJ/kmol]
# ---- route A: closed forms of the slide equations (SI: P kPa, R kJ/(kg K), V m/s, A m2, mdot kg/s)
def cp_of(k, R): return k * R / (k - 1.0)
def T0_of(T, V, cp): return T + V * V / (2000.0 * cp)                         # (1)/(7): V^2/2 in J/kg -> kJ/kg
def P0_of(P, T, T0, k): return P * (T0 / T) ** (k / (k - 1.0))               # (2)
def csound(k, R, T): return math.sqrt(k * R * 1000.0 * T)
def TT0(Ma, k): return 1.0 / (1.0 + (k - 1.0) / 2.0 * Ma * Ma)               # (9)
def PP0(Ma, k): return TT0(Ma, k) ** (k / (k - 1.0))                         # (10)
def AAs(Ma, k): return (1.0 / Ma) * ((2.0 / (k + 1.0)) * (1.0 + (k - 1.0) / 2.0 * Ma * Ma)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (17)
def crit(k): return dict(TT=2.0 / (k + 1.0), PP=(2.0 / (k + 1.0)) ** (k / (k - 1.0)))                                      # (12),(13)
def Ma_of_PP(pr, k): return math.sqrt(2.0 / (k - 1.0) * (pr ** (-(k - 1.0) / k) - 1.0))
def mdot_choked(A, P0, T0, k, R): return A * (P0 * 1000.0) * math.sqrt(k / (R * 1000.0 * T0)) * (2.0 / (k + 1.0)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (16) with P0 in Pa, R in J/(kg K)
# ---- route B: numerical descriptions that do not use (9)-(17)
def flux_max(k):
    """x* = T/T0 maximising the mass flux rho V ~ x^(1/(k-1)) sqrt(1-x) along the isentrope (energy equation V = sqrt(2 cp (T0 - T)))."""
    r = minimize_scalar(lambda x: -(x ** (1.0 / (k - 1.0)) * math.sqrt(1.0 - x)), bounds=(1e-6, 1 - 1e-9), method='bounded', options=dict(xatol=1e-13)); return r.x, -r.fun
def isentrope_T(T, P, Pnew, k):
    s = solve_ivp(lambda x, y: [(k - 1.0) / k], [0.0, math.log(Pnew / P)], [math.log(T)], rtol=1e-11, atol=1e-13); return math.exp(s.y[0, -1])
def c_num(T, k, R):
    """speed of sound as (dP/d rho)_s by central difference on the isentrope P = C rho^k (not sqrt(kRT))."""
    P0_ = 1.0e5; rho = P0_ / (R * 1000.0 * T); C = P0_ / rho ** k; h = 1e-5 * rho; return math.sqrt((C * (rho + h) ** k - C * (rho - h) ** k) / (2 * h))
def x_of_Ma(Ma, k, R=0.287):
    cp = cp_of(k, R); return brentq(lambda T: Ma * c_num(T, k, R) - math.sqrt(max(2000.0 * cp * (1000.0 - T), 0.0)), 1.0, 1000.0 - 1e-9, xtol=1e-12) / 1000.0
k = 1.4
A1s = AAs(0.3, k); A2s = 0.8 * A1s
Ma2_A = brentq(lambda M: AAs(M, k) - A2s, 1e-6, 1.0)                                                # A: brentq on (17), subsonic branch
Ma2_B = solve_ivp(lambda x, y: [y[0] * (1 + (k - 1) / 2 * y[0] ** 2) / (y[0] ** 2 - 1)], [0.0, math.log(0.8)], [0.3], rtol=1e-11, atol=1e-13).y[0, -1]   # B: Hugoniot ODE in ln A
rows = A32[A32[:, 0] <= 1.0]; Ma2_C = float(np.interp(A2s, rows[::-1, 2], rows[::-1, 0]))       # C: linear interpolation in A-32 (subsonic rows)
print(f"A1/A* = {A1s:.4f} (A-32 row: {A32[2, 2]}); A2/A* = {A2s:.4f}")
print(f"Ma2: brentq {Ma2_A:.4f} | ODE {Ma2_B:.4f} | A-32 interpolation {Ma2_C:.4f} | slide prints 0.391")
T2 = 400 * TT0(Ma2_A, k) / TT0(0.3, k); P2 = 100 * PP0(Ma2_A, k) / PP0(0.3, k)
print(f"T2 = {T2:.2f} K (slide 395); P2 = {P2:.2f} kPa (slide 95.7; error of the slide {100 * (95.7 / P2 - 1):+.2f} %)")
Ma = 3.5; lin = float(np.interp(Ma, A32[:, 0], A32[:, 2])); ex = AAs(Ma, k)
print(f"sparse rows: A/A* at Ma 3.5: linear {lin:.2f}, exact {ex:.2f} ({100 * (lin / ex - 1):+.0f} %); P/P0 at Ma 4: linear {np.interp(4.0, A32[:, 0], A32[:, 3]):.5f}, exact {PP0(4.0, k):.5f} ({100 * (np.interp(4.0, A32[:, 0], A32[:, 3]) / PP0(4.0, k) - 1):+.0f} %)")

A1/A* = 2.0351 (A-32 row: 2.0351); A2/A* = 1.6281
Ma2: brentq 0.3887 | ODE 0.3887 | A-32 interpolation 0.3915 | slide prints 0.391
T2 = 395.26 K (slide 395); P2 = 95.91 kPa (slide 95.7; error of the slide -0.22 %)
sparse rows: A/A* at Ma 3.5: linear 9.43, exact 6.79 (+39 %); P/P0 at Ma 4: linear 0.01455, exact 0.00659 (+121 %)


**Đọc kết quả:** A₂/A* = 1,6281; Ma₂ = 0,3887 (brentq và ODE trùng), nội suy A-32 cho 0,3915 nên slide in 0,391. T₂ = 395,26 K (slide 395 đúng), P₂ = 95,91 kPa (slide 95,7 lệch −0,22 %). Với hàng thưa Ma 3 → 5, nội suy sai +39 % cho A/A* tại Ma = 3,5 và +121 % cho P/P₀ tại Ma = 4: chỉ dùng A-32 ở hàng in, còn lại dùng công thức.

## 7. Kiểm bảng A-32: mọi hàng in so với công thức, và lỗi in ρ/ρ₀ tại Ma = 3,0

**Câu hỏi:** các hàng in của A-32 (Ma*, A/A*, P/P₀, ρ/ρ₀, T/T₀) có khớp công thức (9)–(11), (17) trong nửa đơn vị của chữ số cuối không? Ô nào sai?

In [7]:
import math, numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq, minimize_scalar
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A32 = np.array([[0.1, 0.1094, 5.8218, 0.993, 0.995, 0.998], [0.2, 0.2182, 2.9635, 0.9725, 0.9803, 0.9921], [0.3, 0.3257, 2.0351, 0.9395, 0.9564, 0.9823], [0.4, 0.4313, 1.5901, 0.8956, 0.9243, 0.969], [0.5, 0.5345, 1.3398, 0.843, 0.8852, 0.9524], [0.6, 0.6348, 1.1882, 0.784, 0.8405, 0.9328], [0.7, 0.7318, 1.0944, 0.7209, 0.7916, 0.9107], [0.8, 0.8251, 1.0382, 0.656, 0.74, 0.8865], [0.9, 0.9146, 1.0089, 0.5913, 0.687, 0.8606], [1.0, 1.0, 1.0, 0.5283, 0.6339, 0.8333], [1.2, 1.1583, 1.0304, 0.4124, 0.5311, 0.7764], [1.4, 1.2999, 1.1149, 0.3142, 0.4374, 0.7184], [1.6, 1.4254, 1.2502, 0.2353, 0.3557, 0.6614], [1.8, 1.536, 1.439, 0.174, 0.2868, 0.6068], [2.0, 1.633, 1.6875, 0.1278, 0.23, 0.5556], [2.2, 1.7179, 2.005, 0.0935, 0.1841, 0.5081], [2.4, 1.7922, 2.4031, 0.0684, 0.1472, 0.4647], [2.6, 1.8571, 2.896, 0.0501, 0.1179, 0.4252], [2.8, 1.914, 3.5001, 0.0368, 0.0946, 0.3894], [3.0, 1.964, 4.2346, 0.0272, 0.076, 0.3571], [5.0, 2.2361, 25.0, 0.0019, 0.0113, 0.1667]])     # A-32 (k = 1.4) rows: Ma, Ma*, A/A*, P/P0, rho/rho0, T/T0
A17 = np.array([[250.0, 250.05, 0.7329], [260.0, 260.09, 0.8405], [270.0, 270.11, 0.959], [280.0, 280.13, 1.0889], [285.0, 285.14, 1.1584], [290.0, 290.16, 1.2311], [295.0, 295.17, 1.3068], [298.0, 298.18, 1.3543], [300.0, 300.19, 1.386], [305.0, 305.22, 1.4686], [310.0, 310.24, 1.5546], [315.0, 315.27, 1.6442], [320.0, 320.29, 1.7375], [325.0, 325.31, 1.8345], [330.0, 330.34, 1.9352], [340.0, 340.42, 2.149], [350.0, 350.49, 2.379], [360.0, 360.58, 2.626], [370.0, 370.67, 2.892], [380.0, 380.77, 3.176], [390.0, 390.88, 3.481], [400.0, 400.98, 3.806], [410.0, 411.12, 4.153], [420.0, 421.26, 4.522], [430.0, 431.43, 4.915], [440.0, 441.61, 5.332], [450.0, 451.8, 5.775], [460.0, 462.02, 6.245], [470.0, 472.24, 6.742], [480.0, 482.49, 7.268], [490.0, 492.74, 7.824], [500.0, 503.02, 8.411], [510.0, 513.32, 9.031], [520.0, 523.63, 9.684], [530.0, 533.98, 10.37], [540.0, 544.35, 11.1], [550.0, 555.74, 11.86], [560.0, 565.17, 12.66], [570.0, 575.59, 13.5], [580.0, 586.04, 14.38], [590.0, 596.52, 15.31], [600.0, 607.02, 16.28], [610.0, 617.53, 17.3], [620.0, 628.07, 18.36], [630.0, 638.63, 19.84], [640.0, 649.22, 20.64], [650.0, 659.84, 21.86], [660.0, 670.47, 23.13], [670.0, 681.14, 24.46], [680.0, 691.82, 25.85], [690.0, 702.52, 27.29], [700.0, 713.27, 28.8], [710.0, 724.04, 30.38], [720.0, 734.82, 32.02], [730.0, 745.62, 33.72], [740.0, 756.44, 35.5], [750.0, 767.29, 37.35], [760.0, 778.18, 39.27], [780.0, 800.03, 43.35], [800.0, 821.95, 47.75]])     # A-17 air: T [K], h [kJ/kg], Pr
A20 = np.array([[300.0, 9431.0], [310.0, 9807.0], [320.0, 10186.0], [330.0, 10570.0], [340.0, 10959.0], [350.0, 11351.0], [360.0, 11748.0], [370.0, 12148.0], [380.0, 12552.0], [390.0, 12960.0], [400.0, 13372.0], [410.0, 13787.0], [420.0, 14206.0], [430.0, 14628.0], [440.0, 15054.0], [450.0, 15483.0], [460.0, 15916.0], [470.0, 16351.0], [480.0, 16791.0], [490.0, 17232.0], [500.0, 17678.0], [510.0, 18126.0], [520.0, 18576.0], [530.0, 19029.0], [540.0, 19485.0], [550.0, 19945.0], [560.0, 20407.0], [570.0, 20870.0], [580.0, 21337.0], [590.0, 21807.0], [600.0, 22280.0], [610.0, 22754.0], [620.0, 23231.0], [630.0, 23709.0], [640.0, 24190.0], [650.0, 24674.0], [660.0, 25160.0], [670.0, 25648.0], [680.0, 26138.0], [690.0, 26631.0], [700.0, 27125.0], [710.0, 27622.0], [720.0, 28121.0], [730.0, 28622.0], [740.0, 29124.0], [750.0, 29629.0], [760.0, 30135.0], [770.0, 30644.0], [780.0, 31154.0], [790.0, 31665.0], [800.0, 32179.0], [810.0, 32694.0], [820.0, 33212.0], [830.0, 33730.0], [840.0, 34251.0], [850.0, 34773.0], [860.0, 35296.0], [870.0, 35821.0], [880.0, 36347.0], [890.0, 36876.0], [900.0, 37405.0], [910.0, 37935.0], [920.0, 38467.0], [930.0, 39000.0], [940.0, 39535.0], [950.0, 40070.0], [960.0, 40607.0], [970.0, 41145.0], [980.0, 41685.0], [990.0, 42226.0], [1000.0, 42769.0], [1020.0, 43859.0], [1040.0, 44953.0], [1060.0, 46051.0], [1080.0, 47153.0], [1100.0, 48258.0], [1120.0, 49369.0], [1140.0, 50484.0], [1160.0, 51602.0], [1180.0, 52724.0], [1200.0, 53848.0], [1220.0, 54977.0], [1240.0, 56108.0], [1260.0, 57244.0], [1280.0, 58381.0], [1300.0, 59522.0], [1320.0, 60666.0], [1340.0, 61813.0], [1360.0, 62963.0], [1380.0, 64116.0], [1400.0, 65271.0]])     # A-20 CO2: T [K], h [kJ/kmol]
# ---- route A: closed forms of the slide equations (SI: P kPa, R kJ/(kg K), V m/s, A m2, mdot kg/s)
def cp_of(k, R): return k * R / (k - 1.0)
def T0_of(T, V, cp): return T + V * V / (2000.0 * cp)                         # (1)/(7): V^2/2 in J/kg -> kJ/kg
def P0_of(P, T, T0, k): return P * (T0 / T) ** (k / (k - 1.0))               # (2)
def csound(k, R, T): return math.sqrt(k * R * 1000.0 * T)
def TT0(Ma, k): return 1.0 / (1.0 + (k - 1.0) / 2.0 * Ma * Ma)               # (9)
def PP0(Ma, k): return TT0(Ma, k) ** (k / (k - 1.0))                         # (10)
def AAs(Ma, k): return (1.0 / Ma) * ((2.0 / (k + 1.0)) * (1.0 + (k - 1.0) / 2.0 * Ma * Ma)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (17)
def crit(k): return dict(TT=2.0 / (k + 1.0), PP=(2.0 / (k + 1.0)) ** (k / (k - 1.0)))                                      # (12),(13)
def Ma_of_PP(pr, k): return math.sqrt(2.0 / (k - 1.0) * (pr ** (-(k - 1.0) / k) - 1.0))
def mdot_choked(A, P0, T0, k, R): return A * (P0 * 1000.0) * math.sqrt(k / (R * 1000.0 * T0)) * (2.0 / (k + 1.0)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (16) with P0 in Pa, R in J/(kg K)
# ---- route B: numerical descriptions that do not use (9)-(17)
def flux_max(k):
    """x* = T/T0 maximising the mass flux rho V ~ x^(1/(k-1)) sqrt(1-x) along the isentrope (energy equation V = sqrt(2 cp (T0 - T)))."""
    r = minimize_scalar(lambda x: -(x ** (1.0 / (k - 1.0)) * math.sqrt(1.0 - x)), bounds=(1e-6, 1 - 1e-9), method='bounded', options=dict(xatol=1e-13)); return r.x, -r.fun
def isentrope_T(T, P, Pnew, k):
    s = solve_ivp(lambda x, y: [(k - 1.0) / k], [0.0, math.log(Pnew / P)], [math.log(T)], rtol=1e-11, atol=1e-13); return math.exp(s.y[0, -1])
def c_num(T, k, R):
    """speed of sound as (dP/d rho)_s by central difference on the isentrope P = C rho^k (not sqrt(kRT))."""
    P0_ = 1.0e5; rho = P0_ / (R * 1000.0 * T); C = P0_ / rho ** k; h = 1e-5 * rho; return math.sqrt((C * (rho + h) ** k - C * (rho - h) ** k) / (2 * h))
def x_of_Ma(Ma, k, R=0.287):
    cp = cp_of(k, R); return brentq(lambda T: Ma * c_num(T, k, R) - math.sqrt(max(2000.0 * cp * (1000.0 - T), 0.0)), 1.0, 1000.0 - 1e-9, xtol=1e-12) / 1000.0
k = 1.4; bad = []; n = 0
for Ma, Mas, aa, pp, rr, tt in A32:
    f_ = dict(Mas=Ma * math.sqrt((k + 1) / (2 + (k - 1) * Ma ** 2)), aa=AAs(Ma, k), pp=PP0(Ma, k), rr=TT0(Ma, k) ** (1 / (k - 1)), tt=TT0(Ma, k))
    for key, val in (('Mas', Mas), ('aa', aa), ('pp', pp), ('rr', rr), ('tt', tt)):
        n += 1; tol = 5.1e-4 if (key == 'aa' and Ma == 5.0) else 5.1e-5
        if abs(val - f_[key]) > tol: bad.append((float(Ma), key, float(val), round(float(f_[key]), 5)))
print(n, "printed cells compared; outside half a unit of the last digit:", bad)

105 printed cells compared; outside half a unit of the last digit: [(3.0, 'rr', 0.076, 0.07623)]


**Đọc kết quả:** trong 21 hàng (Ma > 0) × 5 cột = 105 ô chỉ một ô lệch: Ma = 3,0, ρ/ρ₀ in 0,0760, công thức cho 0,07623 (đúng 0,0762). Đây là lỗi in của bảng (thư viện P3 đánh dấu suspect); câu hỏi của module không dùng ô này.

## 8. Vòi hội tụ–phân kỳ (Cengel Ví dụ 17–7, 17–135) và c_p biến thiên (17–11, 17–20c)

**Câu hỏi:** (a) Ví dụ 17–7: P₀ = 1 MPa, T₀ = 800 K, A* = 20 cm², Ma_e = 2: A_e, V_e, ṁ; 17–135: heli, A* và A_e. (b) Bài 17–11: T tĩnh khi P₀ = 0,6 MPa, T₀ = 673,15 K, V = 570 m/s với c_p = 1,005 so với bảng A-17 (đáp án in 518,6 K). (c) 17–20c: V₂ của CO₂ khi c_p biến thiên (A-20).

In [8]:
import math, numpy as np
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq, minimize_scalar
# ---- tables read from the Appendix 1 page images (data/props/*.json), embedded so that the notebook is self-contained
A32 = np.array([[0.1, 0.1094, 5.8218, 0.993, 0.995, 0.998], [0.2, 0.2182, 2.9635, 0.9725, 0.9803, 0.9921], [0.3, 0.3257, 2.0351, 0.9395, 0.9564, 0.9823], [0.4, 0.4313, 1.5901, 0.8956, 0.9243, 0.969], [0.5, 0.5345, 1.3398, 0.843, 0.8852, 0.9524], [0.6, 0.6348, 1.1882, 0.784, 0.8405, 0.9328], [0.7, 0.7318, 1.0944, 0.7209, 0.7916, 0.9107], [0.8, 0.8251, 1.0382, 0.656, 0.74, 0.8865], [0.9, 0.9146, 1.0089, 0.5913, 0.687, 0.8606], [1.0, 1.0, 1.0, 0.5283, 0.6339, 0.8333], [1.2, 1.1583, 1.0304, 0.4124, 0.5311, 0.7764], [1.4, 1.2999, 1.1149, 0.3142, 0.4374, 0.7184], [1.6, 1.4254, 1.2502, 0.2353, 0.3557, 0.6614], [1.8, 1.536, 1.439, 0.174, 0.2868, 0.6068], [2.0, 1.633, 1.6875, 0.1278, 0.23, 0.5556], [2.2, 1.7179, 2.005, 0.0935, 0.1841, 0.5081], [2.4, 1.7922, 2.4031, 0.0684, 0.1472, 0.4647], [2.6, 1.8571, 2.896, 0.0501, 0.1179, 0.4252], [2.8, 1.914, 3.5001, 0.0368, 0.0946, 0.3894], [3.0, 1.964, 4.2346, 0.0272, 0.076, 0.3571], [5.0, 2.2361, 25.0, 0.0019, 0.0113, 0.1667]])     # A-32 (k = 1.4) rows: Ma, Ma*, A/A*, P/P0, rho/rho0, T/T0
A17 = np.array([[250.0, 250.05, 0.7329], [260.0, 260.09, 0.8405], [270.0, 270.11, 0.959], [280.0, 280.13, 1.0889], [285.0, 285.14, 1.1584], [290.0, 290.16, 1.2311], [295.0, 295.17, 1.3068], [298.0, 298.18, 1.3543], [300.0, 300.19, 1.386], [305.0, 305.22, 1.4686], [310.0, 310.24, 1.5546], [315.0, 315.27, 1.6442], [320.0, 320.29, 1.7375], [325.0, 325.31, 1.8345], [330.0, 330.34, 1.9352], [340.0, 340.42, 2.149], [350.0, 350.49, 2.379], [360.0, 360.58, 2.626], [370.0, 370.67, 2.892], [380.0, 380.77, 3.176], [390.0, 390.88, 3.481], [400.0, 400.98, 3.806], [410.0, 411.12, 4.153], [420.0, 421.26, 4.522], [430.0, 431.43, 4.915], [440.0, 441.61, 5.332], [450.0, 451.8, 5.775], [460.0, 462.02, 6.245], [470.0, 472.24, 6.742], [480.0, 482.49, 7.268], [490.0, 492.74, 7.824], [500.0, 503.02, 8.411], [510.0, 513.32, 9.031], [520.0, 523.63, 9.684], [530.0, 533.98, 10.37], [540.0, 544.35, 11.1], [550.0, 555.74, 11.86], [560.0, 565.17, 12.66], [570.0, 575.59, 13.5], [580.0, 586.04, 14.38], [590.0, 596.52, 15.31], [600.0, 607.02, 16.28], [610.0, 617.53, 17.3], [620.0, 628.07, 18.36], [630.0, 638.63, 19.84], [640.0, 649.22, 20.64], [650.0, 659.84, 21.86], [660.0, 670.47, 23.13], [670.0, 681.14, 24.46], [680.0, 691.82, 25.85], [690.0, 702.52, 27.29], [700.0, 713.27, 28.8], [710.0, 724.04, 30.38], [720.0, 734.82, 32.02], [730.0, 745.62, 33.72], [740.0, 756.44, 35.5], [750.0, 767.29, 37.35], [760.0, 778.18, 39.27], [780.0, 800.03, 43.35], [800.0, 821.95, 47.75]])     # A-17 air: T [K], h [kJ/kg], Pr
A20 = np.array([[300.0, 9431.0], [310.0, 9807.0], [320.0, 10186.0], [330.0, 10570.0], [340.0, 10959.0], [350.0, 11351.0], [360.0, 11748.0], [370.0, 12148.0], [380.0, 12552.0], [390.0, 12960.0], [400.0, 13372.0], [410.0, 13787.0], [420.0, 14206.0], [430.0, 14628.0], [440.0, 15054.0], [450.0, 15483.0], [460.0, 15916.0], [470.0, 16351.0], [480.0, 16791.0], [490.0, 17232.0], [500.0, 17678.0], [510.0, 18126.0], [520.0, 18576.0], [530.0, 19029.0], [540.0, 19485.0], [550.0, 19945.0], [560.0, 20407.0], [570.0, 20870.0], [580.0, 21337.0], [590.0, 21807.0], [600.0, 22280.0], [610.0, 22754.0], [620.0, 23231.0], [630.0, 23709.0], [640.0, 24190.0], [650.0, 24674.0], [660.0, 25160.0], [670.0, 25648.0], [680.0, 26138.0], [690.0, 26631.0], [700.0, 27125.0], [710.0, 27622.0], [720.0, 28121.0], [730.0, 28622.0], [740.0, 29124.0], [750.0, 29629.0], [760.0, 30135.0], [770.0, 30644.0], [780.0, 31154.0], [790.0, 31665.0], [800.0, 32179.0], [810.0, 32694.0], [820.0, 33212.0], [830.0, 33730.0], [840.0, 34251.0], [850.0, 34773.0], [860.0, 35296.0], [870.0, 35821.0], [880.0, 36347.0], [890.0, 36876.0], [900.0, 37405.0], [910.0, 37935.0], [920.0, 38467.0], [930.0, 39000.0], [940.0, 39535.0], [950.0, 40070.0], [960.0, 40607.0], [970.0, 41145.0], [980.0, 41685.0], [990.0, 42226.0], [1000.0, 42769.0], [1020.0, 43859.0], [1040.0, 44953.0], [1060.0, 46051.0], [1080.0, 47153.0], [1100.0, 48258.0], [1120.0, 49369.0], [1140.0, 50484.0], [1160.0, 51602.0], [1180.0, 52724.0], [1200.0, 53848.0], [1220.0, 54977.0], [1240.0, 56108.0], [1260.0, 57244.0], [1280.0, 58381.0], [1300.0, 59522.0], [1320.0, 60666.0], [1340.0, 61813.0], [1360.0, 62963.0], [1380.0, 64116.0], [1400.0, 65271.0]])     # A-20 CO2: T [K], h [kJ/kmol]
# ---- route A: closed forms of the slide equations (SI: P kPa, R kJ/(kg K), V m/s, A m2, mdot kg/s)
def cp_of(k, R): return k * R / (k - 1.0)
def T0_of(T, V, cp): return T + V * V / (2000.0 * cp)                         # (1)/(7): V^2/2 in J/kg -> kJ/kg
def P0_of(P, T, T0, k): return P * (T0 / T) ** (k / (k - 1.0))               # (2)
def csound(k, R, T): return math.sqrt(k * R * 1000.0 * T)
def TT0(Ma, k): return 1.0 / (1.0 + (k - 1.0) / 2.0 * Ma * Ma)               # (9)
def PP0(Ma, k): return TT0(Ma, k) ** (k / (k - 1.0))                         # (10)
def AAs(Ma, k): return (1.0 / Ma) * ((2.0 / (k + 1.0)) * (1.0 + (k - 1.0) / 2.0 * Ma * Ma)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (17)
def crit(k): return dict(TT=2.0 / (k + 1.0), PP=(2.0 / (k + 1.0)) ** (k / (k - 1.0)))                                      # (12),(13)
def Ma_of_PP(pr, k): return math.sqrt(2.0 / (k - 1.0) * (pr ** (-(k - 1.0) / k) - 1.0))
def mdot_choked(A, P0, T0, k, R): return A * (P0 * 1000.0) * math.sqrt(k / (R * 1000.0 * T0)) * (2.0 / (k + 1.0)) ** ((k + 1.0) / (2.0 * (k - 1.0)))   # (16) with P0 in Pa, R in J/(kg K)
# ---- route B: numerical descriptions that do not use (9)-(17)
def flux_max(k):
    """x* = T/T0 maximising the mass flux rho V ~ x^(1/(k-1)) sqrt(1-x) along the isentrope (energy equation V = sqrt(2 cp (T0 - T)))."""
    r = minimize_scalar(lambda x: -(x ** (1.0 / (k - 1.0)) * math.sqrt(1.0 - x)), bounds=(1e-6, 1 - 1e-9), method='bounded', options=dict(xatol=1e-13)); return r.x, -r.fun
def isentrope_T(T, P, Pnew, k):
    s = solve_ivp(lambda x, y: [(k - 1.0) / k], [0.0, math.log(Pnew / P)], [math.log(T)], rtol=1e-11, atol=1e-13); return math.exp(s.y[0, -1])
def c_num(T, k, R):
    """speed of sound as (dP/d rho)_s by central difference on the isentrope P = C rho^k (not sqrt(kRT))."""
    P0_ = 1.0e5; rho = P0_ / (R * 1000.0 * T); C = P0_ / rho ** k; h = 1e-5 * rho; return math.sqrt((C * (rho + h) ** k - C * (rho - h) ** k) / (2 * h))
def x_of_Ma(Ma, k, R=0.287):
    cp = cp_of(k, R); return brentq(lambda T: Ma * c_num(T, k, R) - math.sqrt(max(2000.0 * cp * (1000.0 - T), 0.0)), 1.0, 1000.0 - 1e-9, xtol=1e-12) / 1000.0
k, R = 1.4, 0.287; xs, fs = flux_max(k); cp = cp_of(k, R)
# (a) Example 17-7: throat by flux maximum, exit by brentq on the energy equation (no use of eq. 17)
P0, T0, As = 1000.0, 800.0, 20.0
mdot = As * 1e-4 * (P0 / (R * T0)) * xs ** (1 / (k - 1)) * math.sqrt(2000 * cp * T0 * (1 - xs)); xe = x_of_Ma(2.0, k, R)
fe = xe ** (1 / (k - 1)) * math.sqrt(1 - xe); Ae = As * fs / fe; Ve = 2.0 * c_num(T0 * xe, k, R)
print(f"17-7: T* = {xs * T0:.1f} K; mdot = {mdot:.3f} kg/s (book 2.86); A_e = {Ae:.2f} cm2 (33.75); V_e = {Ve:.1f} m/s (845.1-845.2); closed form A_e = {As * AAs(2.0, k):.2f}")
kH, RH = 1.667, 2.0769; xsH, fsH = flux_max(kH); AsH = 0.25 / ((1000.0 / (RH * 500.0)) * xsH ** (1 / (kH - 1)) * math.sqrt(2000 * cp_of(kH, RH) * 500.0 * (1 - xsH))) * 1e4
xeH = (100.0 / 1000.0) ** ((kH - 1) / kH); AeH = AsH * fsH / (xeH ** (1 / (kH - 1)) * math.sqrt(1 - xeH)); print(f"17-135 helium: A* = {AsH:.3f} cm2 (printed 3.51); A_e = {AeH:.3f} cm2 (printed 5.84)")
# (b) 17-11
h0 = float(np.interp(673.15, A17[:, 0], A17[:, 1])); hs = h0 - 570.0 ** 2 / 2000.0; Tv = float(np.interp(hs, A17[:, 1], A17[:, 0]))
Pv = 600.0 * float(np.interp(Tv, A17[:, 0], A17[:, 2])) / float(np.interp(673.15, A17[:, 0], A17[:, 2])); Tc = 673.15 - 570.0 ** 2 / 2010.0
print(f"17-11: constant cp {Tc:.1f} K, {600 * (Tc / 673.15) ** 3.5:.1f} kPa | A-17 {Tv:.1f} K, {Pv:.1f} kPa | printed 518.6 K, 0.23 MPa")
# (c) 17-20c: CO2 1200 K -> 400 K, V1 = 50 m/s
dh = (np.interp(1200.0, A20[:, 0], A20[:, 1]) - np.interp(400.0, A20[:, 0], A20[:, 1])) / 44.01; V2c = math.sqrt(50.0 ** 2 + 2000.0 * 0.846 * 800.0); V2v = math.sqrt(50.0 ** 2 + 2000.0 * dh)
print(f"17-20c: dh = {dh:.1f} kJ/kg (constant cp 0.846 x 800 = {0.846 * 800:.1f}); V2 = {V2c:.0f} m/s (constant cp) vs {V2v:.0f} m/s (A-20): {100 * (V2v / V2c - 1):+.0f} %")

17-7: T* = 666.7 K; mdot = 2.858 kg/s (book 2.86); A_e = 33.75 cm2 (33.75); V_e = 845.2 m/s (845.1-845.2); closed form A_e = 33.75
17-135 helium: A* = 3.508 cm2 (printed 3.51); A_e = 5.845 cm2 (printed 5.84)
17-11: constant cp 511.5 K, 229.5 kPa | A-17 518.5 K, 231.0 kPa | printed 518.6 K, 0.23 MPa
17-20c: dh = 919.7 kJ/kg (constant cp 0.846 x 800 = 676.8); V2 = 1165 m/s (constant cp) vs 1357 m/s (A-20): +17 %


**Đọc kết quả:** (a) ṁ = 2,858 kg/s, A_e = 33,75 cm², V_e = 845,1 m/s khớp in 2,86; 33,75; 845,1–845,2; heli A* = 3,508 cm², A_e = 5,845 cm² (in 3,51; 5,84). Cực đại số của ρV và brentq trên phương trình năng lượng cho cùng kết quả với công thức (16), (17). (b) c_p = 1,005 cho 511,5 K (−1,4 %), bảng A-17 cho 518,5 K: đáp án in 518,6 K cần c_p biến thiên. (c) với A-20, V₂ lớn hơn 17 % so với c_p = 0,846 không đổi: giả thiết c_p không đổi kém khi nhiệt độ đổi 800 K.